# 📝 과제 02: 의약품 상담 도우미의 검색 전략과 아키텍처 종합 선정

## 만들 결과

약국 체인 본사의 **의약품 상담 검색기**에서 질문 유형마다 답하는 길(검색 전략 또는 구조화 질의)을 나누고, 검색으로 답하는 질문에 쓸 검색 전략과 벡터 저장 형식·저장소를 정한 뒤, 네 축으로 최종 구성 하나를 고릅니다. 임베딩 모델은 과제 01이 고른 KURE-v1 하나만 씁니다.

- 질문 다섯 유형을 검색 전략으로 답할 유형과 구조화 질의로 답할 유형으로 나눕니다.
- LangChain으로 세 검색기(`bm25`, `dense`, `hybrid`)를 만들고, 검색으로 답하는 질문 36개로 유형 × 전략 정확도(Recall@5) 표를 만듭니다.
- 유형마다 다른 전략을 쓰는 라우터를 둘 만큼 정확도가 오르는지 전략 하나로 처리할 때와 비교합니다.
- 메모리 예산 안에 벡터를 담을 저장 형식(정밀도)과, 서비스가 보낼 질의를 처리하는 저장소를 고릅니다.
- 세 전략 후보를 네 축(정확도·지연·비용·저장 공간) 후보표로 만들어 제약·지배·고정 기준 점수로 1위를 고르고, 가정을 바꾼 시나리오와 가중치로 1위가 바뀌는 지점을 찾습니다.
- 결정을 ADR로 남깁니다(ADR-002 검색 전략, ADR-003 저장 형식과 벡터 저장소, ADR-004 최종 아키텍처). 과제 01의 ADR-001에 이어집니다.

교안 02의 질문 유형 분류, 유형 × 전략 표, 필요한 질의로 저장소 고르기, 저장 형식 정하기, 네 축 종합 선정, ADR을 적용합니다. 의학적 조언이 아닌 수업용 검색 실습이며, 실제 복용은 의사·약사와 상의해야 합니다.

**선정 기준**

서비스 가정 (과제 01의 가정에 저장소·규모·장비 가정을 더합니다)
- 약국 체인 본사 검색 서버 하나가 매장 약사의 검색을 처리합니다.
- 본사는 이미 Neo4j로 제품·성분 그래프를 운영하고, PostgreSQL은 운영하지 않습니다.
- 품목 5만 건, 매장 2,000곳이 하루 50건씩 묻습니다(월 질문 300만 건이 하루 종일 들어옴).
- 검색 서버에서 벡터에 줄 수 있는 메모리는 60MB입니다.
- 검색 서버는 GPU 없는 CPU 장비입니다. 질문을 KURE-v1로 임베딩하는 전략(`dense`, `hybrid`)은 질문 임베딩 장비(시간당 $0.35, 월 730시간)를 늘 켜 둡니다. BM25만 쓰는 전략은 질문을 검색 서버에서 바로 계산해 이 장비를 늘 켜 둘 필요가 없습니다. 문서 색인 비용은 어느 전략이든 색인에 걸린 시간을 같은 장비 가격으로 환산합니다.

팀이 합의한 고정값
- 라우터: 유형별로 고른 전략의 맞힌 문항이 가장 좋은 전략 하나보다 2문항 이상 많을 때만 질문 유형을 고르는 라우터를 둡니다(1문항 이내면 같은 수준, 과제 01과 같은 규칙).
- 저장 형식(정밀도): 5만 건 벡터 크기가 60MB 이하인 정밀도만 남기고, 맞힌 문항이 가장 많은 것과 1문항 이내면 크기가 작은 것
- 저장소: 필요한 질의를 모두 한 질의로 처리하는 저장소. 둘 이상이면 이미 운영하는 DB
- 제약: 지연 500ms 이하, 저장 2GB 이하
- 고정 기준(받아들일 수 있는 최대치): 지연 500ms, 월 비용 $800, 저장 2GB
- 가중치: 정확도 0.7, 지연 0.1, 비용 0.1, 저장 0.1

교안과 다른 점:
- 교안 본실습(여행지)은 관광지 100만 건·월 질문 1만 건이라 비용이 작았고, 회사가 PostgreSQL을 운영해 pgvector를 골랐습니다. 지연은 이 PC에서 측정한 검색 시간을 썼습니다.
- 이 과제는 월 질문이 300만 건이라 질문 임베딩 장비를 늘 켜 두는 비용이 KURE-v1을 쓰는 전략에만 붙고, 지연은 운영 장비(CPU)의 가정 시간을 씁니다.
- 운영 중인 DB는 Neo4j이고, 벡터 메모리 예산(60MB)이 있어 저장 형식도 함께 정합니다. 가중치는 교안과 같습니다.
- 저장 축은 교안 6-1과 같은 식(원래 벡터 × 교안 4-2에서 측정한 배수 3.0 + BM25 색인)으로 계산하지만 뜻이 다릅니다. 교안은 float32를 그대로 써서 이 값이 검색에 쓰는 메모리였고, 이 과제는 binary 인덱스라 메모리에는 binary 사본만 올리고 원래 벡터는 재점수용으로 함께 둡니다. 그래서 같은 식의 값을 메모리가 아니라 검색에 두는 저장 공간 전체로 읽고, 메모리는 4번의 60MB 예산으로 따로 봅니다.

<img src="images/gwaje02_overview.png" width="1050" alt="의약품 상담 도우미의 검색 전략·저장소·종합 선정 흐름. 질문 다섯 유형을 검색 전략으로 답할 유형과 구조화 질의로 답할 유형으로 나누고, 세 검색기(bm25·dense·hybrid)의 정확도를 유형마다 비교해 전략을 고른 뒤, 메모리 예산으로 저장 형식(binary)을 정하고 서비스가 보낼 질의로 저장소를 고릅니다. 세 후보를 네 축의 후보표에 놓고 제약으로 거르고 지배당한 후보를 지우고 가중 점수로 1위를 고릅니다.">

## 준비: 데이터와 제공 코드

- **자료:** `data/과제/`의 의약품 안내문 394건(`docs.jsonl`)과 평가 질문 60개(`questions.json`, 묘사·이름·조건·관계·집계 각 12). 안내문 출처는 식품의약품안전처 「의약품개요정보(e약은요)」(공공누리 출처표시)입니다.
- **앞 과제 결과:** 과제 01의 `output/gwaje01_embedding_choice.json`에서 고른 모델을 읽습니다. 건너뛰었으면 `data/과제/`의 같은 이름 사본을 읽습니다.
- **순서:** 답하는 길 나누기 -> 세 검색기와 유형 × 전략 표 -> 라우터 비교 -> ADR-002 -> 저장 형식과 저장소 -> ADR-003 -> 후보표 -> 선정과 환율 -> 시나리오 -> 민감도 -> ADR-004
- **제공:** 환경 설정, 자료 읽기, KURE-v1 임베딩 모델과 문서 목록(`drug_documents`), 평가·낱말 함수(`recall_at_k`, `kiwi_tokenize`), 정밀도 함수(`lower_precision`, `storage_mb`, `top5_ids`, `count_found`), 후보표 함수(`monthly_cost`, `build_candidates`), 선정 함수(`apply_constraints`, `remove_dominated`, `score_candidates`, `run_selection`, `exchange_rates`, `check_sensitivity`)
- **작성:** `[작성]` 셀의 `...`를 채우고 바로 아래 `[자가채점]` 셀을 실행하세요. 이 과제는 외부 API를 부르지 않습니다.
- **로컬 모델:** 교안에서 내려받은 KURE-v1을 씁니다. GPU 없는 PC에서는 2번의 문서 394건 임베딩에 몇 분이 걸립니다.
- **DB 접속 없음:** 4번의 저장소는 기능 표로 판정하므로 이 과제는 Neo4j나 Supabase에 접속하지 않습니다.

In [ ]:
# 실습에 필요한 라이브러리를 불러옵니다. 나머지는 처음 쓰는 셀에서 불러옵니다.
import json
from pathlib import Path

import numpy as np
import pandas as pd

# 단원 폴더(material_dir) 아래 data·output 폴더 경로
material_dir = Path(".").resolve()
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)

# 과제 자료 폴더가 있는지
print("과제 자료 폴더 있음:", (data_dir / "과제").exists())

In [ ]:
def read_json(name):
    """data 폴더의 JSON 파일을 목록 또는 딕셔너리로 읽습니다."""
    return json.loads((data_dir / name).read_text(encoding="utf-8"))


# 의약품 안내문(JSON Lines라 한 줄씩 읽기)과 평가 질문
docs = [json.loads(line) for line in (data_dir / "과제" / "docs.jsonl").read_text(encoding="utf-8").splitlines()]
questions = read_json("과제/questions.json")
drug_ids = [doc["doc_id"] for doc in docs]  # 원문 ID(안내문 순서)
print("안내문:", len(docs), "| 질문:", len(questions))

# 유형별 질문 수
question_table = pd.DataFrame(questions)
display(question_table["question_type"].value_counts().rename("질문 수").to_frame())

과제 01의 결과 파일을 읽습니다. 과제 01을 마쳤으면 바로 아래 셀(`output/`), 건너뛰었으면 그다음 셀(미리 넣어 둔 `data/과제/` 사본)을 실행합니다. 두 셀 가운데 하나만 실행합니다.

In [ ]:
# 과제 01을 마쳤으면 이 셀을 실행합니다. 아래 셀과 둘 중 하나만 실행합니다.
embedding_choice = json.loads((output_dir / "gwaje01_embedding_choice.json").read_text(encoding="utf-8"))
print("과제 01이 고른 모델:", embedding_choice["model"])

In [ ]:
# 과제 01을 건너뛰었으면 이 셀로 미리 넣어 둔 같은 이름의 사본을 읽습니다.
embedding_choice = read_json("과제/gwaje01_embedding_choice.json")
print("과제 01이 고른 모델:", embedding_choice["model"])

과제 01이 고른 KURE-v1을 교안 02와 같은 `HuggingFaceEmbeddings`로 불러오고, 안내문마다 LangChain `Document`를 만듭니다. 원문 ID는 metadata의 `source_id`에 담아 검색 결과에서 바로 읽습니다.

In [ ]:
from kiwipiepy import Kiwi
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

kiwi = Kiwi()


# 검색 품질 평가 단원의 함수(predicted: 검색 결과 ID 순위순, relevant: 정답 ID 목록)
def recall_at_k(predicted, relevant, k):
    """전체 관련 문서 중 상위 k개가 찾아낸 비율(관련 수 / 전체 관련 수)."""
    hits = sum(1 for p in predicted[:k] if p in relevant)
    return hits / len(relevant)


def kiwi_tokenize(text):
    """문서와 질문에서 명사·외국어·숫자를 같은 기준으로 추출합니다."""
    return [token.form.lower() for token in kiwi.tokenize(text.replace("･", "·"))
            if token.tag.startswith("N") or token.tag in {"SL", "SN"}]


# 안내문마다 Document 하나(본문은 제목 + 줄바꿈 + 본문, 과제 01과 같음)
drug_documents = []
for doc in docs:
    drug_documents.append(Document(id=doc["doc_id"], page_content=doc["title"] + "\n" + doc["text"],
                                   metadata={"source_id": doc["doc_id"]}))

# 과제 01이 고른 임베딩 모델(벡터 길이를 1로 맞춰 거리 순서가 코사인 유사도 순서와 같음)
embedding_model = HuggingFaceEmbeddings(model_name="nlpai-lab/KURE-v1", encode_kwargs={"normalize_embeddings": True})
print("문서:", len(drug_documents), "| 임베딩 모델:", embedding_model.model_name)

## 1. 질문 유형마다 검색 전략으로 답할지 구조화 질의로 답할지 나누세요

**배경**: 문서를 찾기만 하면 답이 나오는 질문만 검색 전략의 정확도로 비교할 수 있으므로, 측정하기 전에 유형마다 답하는 길을 나눠야 합니다.

| 유형 | 이 평가셋의 예시 질문 | 답에 필요한 근거 |
|---|---|---|
| 묘사 | 손바닥과 팔 밑이 늘 축축하게 젖어 고민인 사람이 밤에 바르는 약은? | 증상을 안내문과 다른 낱말로 설명한 문장이 가리키는 안내문 한두 건 |
| 이름 | 베르니케 뇌병증에 쓰는 비타민 알약은? | 그 안내문에만 있는 고유한 표현이 든 안내문 한두 건 |
| 조건 | 붙이는 제품 가운데 배를 타기 몇 시간 전에 귀 뒤에 붙여 두는 것은? | 제형 조건(`filter`)에 맞는 안내문 가운데 내용이 맞는 한두 건 |
| 관계 | 판토-에이내복액 먹는 중인데, 이 약 안내문이 함께 먹지 말라고 한 성분이 든 캡슐 제품은? | 기준 제품(`anchor_text`) 안내문 -> 함께 먹지 말라고 한 성분 -> 그 성분을 담은 다른 제품으로 이어지는 관계 두 단계(`drug_ingredients.csv`) |
| 집계 | 바르는 제형 가운데 효능군이 피부인 제품은 몇 개야? | 제형·효능군 조건에 맞는 안내문 전부(질문마다 상위 5개보다 많음) |

**요구사항**
- **`answer_path`**: dict. 키는 다섯 유형 이름(`묘사`, `이름`, `조건`, `관계`, `집계`), 값은 `"검색 전략"` 또는 `"구조화 질의"`
- **`search_types`**: list[str]. `answer_path`의 값이 `"검색 전략"`인 유형을 위 표의 순서대로 담은 리스트

**확인 기준**: `search_types`에 든 유형의 질문은 모두 36개입니다. 2번은 이 36개만 검색해 정확도를 측정합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 검색은 질문과 가까운 안내문 몇 건(상위 5개)을 돌려줍니다. 그 몇 건 안에 근거가 다 들어오면 검색 전략, 관계를 따라가거나 조건에 맞는 것을 모두 세야 하면 구조화 질의입니다.

세부구현:
1. 표의 '답에 필요한 근거' 열을 유형마다 읽고 값을 정해 answer_path에 담는다.
2. 표의 유형 순서대로 돌며 값이 "검색 전략"인 유형만 search_types에 더한다.
```

</details>

In [ ]:
# [작성] 유형마다 답하는 길을 정하세요.
# 1. 다섯 유형을 키로, "검색 전략" 또는 "구조화 질의"를 값으로
answer_path = ...

# 2. 검색 전략으로 답하는 유형(표 순서)
search_types = ...
print("검색으로 답하는 유형:", search_types)

In [ ]:
# [자가채점] 키와 값, 유형마다 고른 길, search_types와 질문 수를 확인합니다.
import hashlib

type_order = ["묘사", "이름", "조건", "관계", "집계"]
assert isinstance(answer_path, dict), "answer_path에 dict를 담으세요. `...`를 채웠는지도 확인하세요."
assert set(answer_path) == set(type_order), "answer_path의 키는 다섯 유형 이름(묘사, 이름, 조건, 관계, 집계)입니다."

# 유형마다 정답 값의 지문(정답을 그대로 적지 않으려고 "유형|값"의 sha256 앞 12글자만 둠)
path_fingerprints = {"묘사": "dbe0c2ccaf2b", "이름": "d4f8b95214de", "조건": "3ffd9e51d876", "관계": "1a930d3f4f0e", "집계": "fbbad4eff082"}
for question_type in type_order:
    value = answer_path[question_type]
    assert value in ["검색 전략", "구조화 질의"], f"{question_type}: 값은 \"검색 전략\" 또는 \"구조화 질의\"입니다."
    digest = hashlib.sha256(f"{question_type}|{value}".encode("utf-8")).hexdigest()[:12]
    assert digest == path_fingerprints[question_type], f"{question_type}: 표의 '답에 필요한 근거'를 다시 보세요. 상위 5개 검색 결과 안에 근거가 다 들어오는지, 관계를 따라가거나 모두 세야 하는지 따져 보세요."

# search_types: "검색 전략"인 유형만 빠짐없이, 표 순서대로 한 번씩
assert isinstance(search_types, list), "search_types에 유형 이름 리스트를 담으세요."
for question_type in type_order:
    if question_type in search_types:
        assert answer_path[question_type] == "검색 전략", f"{question_type}: answer_path 값이 \"검색 전략\"인 유형만 search_types에 담으세요."
    else:
        assert answer_path[question_type] != "검색 전략", f"{question_type}: answer_path 값이 \"검색 전략\"인 유형은 빠짐없이 search_types에 담으세요."
positions = [type_order.index(question_type) for question_type in search_types]
assert positions == sorted(set(positions)), "search_types는 표의 유형 순서대로 한 번씩만 담으세요."
search_question_count = int(question_table["question_type"].isin(search_types).sum())
assert search_question_count == 36, f"search_types의 질문은 36개여야 합니다(지금 {search_question_count}개)."
print("답하는 길 확인 완료 | 측정할 질문:", search_question_count, "개")

**구조화 질의로 보내는 두 유형**
- 관계: Text2Cypher가 기준 제품 이름으로 제품 노드를 찾고 `(기준 제품)-[:WARNS]->(성분)<-[:CONTAINS]-(다른 제품)` 경로를 따라가는 Cypher를 만듭니다. 성분 관계는 `drug_ingredients.csv`(`CONTAINS` 237건, `WARNS` 77건)에 있습니다.
- 집계: Text-to-SQL이나 Text2Cypher가 제형·효능군 조건에 맞는 행을 세는 질의(`count`)를 만듭니다. 정답 문서가 질문마다 5건보다 많아 상위 5개를 돌려주는 검색으로는 셀 수 없습니다.
- 두 유형 모두 질문마다 LLM이 질의를 만들어야 해 이 과제에서는 측정하지 않고, 4번에서 저장소가 그 질의를 받을 수 있는지만 판정합니다.
- 조건 질문의 제형 조건도 질문에서 뽑아야 필터로 붙일 수 있습니다. 그 일은 Self-Query가 LLM으로 하므로, 2번은 조건 질문을 필터 없이 검색하고 필터를 지원하는지는 4번 저장소 판정에서 봅니다.

## 2. 세 검색기를 LangChain으로 만들고 유형 × 전략 표를 만드세요

### 2-1. 세 검색기 만들기

**배경**: 세 전략을 같은 문서·같은 설정으로 비교하려면, 교안 02의 3-1처럼 LangChain 검색기 셋을 같은 `drug_documents`로 만들어야 합니다.

| 전략 | 찾는 방법 |
|---|---|
| `bm25` | 질문 낱말이 문서에 자주 나오고 전체에서 드물수록 높은 점수(어휘 검색) |
| `dense` | 질문·문서 KURE-v1 임베딩의 유사도(의미 검색) |
| `hybrid` | `bm25`와 `dense`의 순위를 RRF로 합침 |

**요구사항**
- **`bm25`**: `BM25Retriever`. `drug_documents`로 만들고 `preprocess_func`는 `kiwi_tokenize`, 상위 10개(`k=10`)
- **`vector_store`**: `Chroma` 메모리 컬렉션(이름 `"gwaje58_drugs"`, `embedding_function`은 `embedding_model`)에 `drug_documents` 394건을 넣은 것. 다시 실행해도 394건이 되도록 넣기 전에 `reset_collection()`으로 비웁니다.
- **`dense`**: `vector_store`의 검색기, 상위 10개(`search_kwargs={"k": 10}`)
- **`hybrid`**: `EnsembleRetriever`. `retrievers=[bm25, dense]`, `weights=[0.5, 0.5]`, `c=60`, `id_key="source_id"`
- **`retrievers`**: dict. 키 `"bm25"`, `"dense"`, `"hybrid"`(이 순서), 값은 위 세 검색기
- **`index_seconds`**: dict. 키 `"BM25"`, `"KURE-v1"`, 값은 BM25 색인 시간과 KURE-v1 임베딩·적재 시간(초). 시간을 측정하는 줄은 골격에 있습니다(5번 비용 계산에 씀).

**확인 기준**: 자가채점 셀이 검색기 설정과 컬렉션 건수(394)를 확인하고, 질문 하나를 세 검색기로 검색해 10건 이상 돌아오는지 봅니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 02의 2-3·3-1처럼 BM25 색인과 Chroma 저장소를 먼저 만들고, 그 둘로 dense와 hybrid를 만든다.

세부구현:
1. BM25Retriever.from_documents에 문서 목록, 낱말 나누기 함수, 상위 개수를 넘긴다.
2. Chroma에 컬렉션 이름과 임베딩 모델을 주어 만들고, reset_collection으로 비운 뒤 add_documents로 문서를 넣는다.
3. 저장소의 as_retriever에 search_kwargs로 상위 개수를 준다.
4. EnsembleRetriever에 검색기 두 개(bm25 먼저), 가중치, RRF 상수 c, 같은 문서를 알아볼 metadata 키를 넘긴다.
5. 세 검색기를 이름 순서대로 dict에 담는다.
```

</details>

In [ ]:
# [작성] 세 검색기를 만드세요.
import time

from langchain_chroma import Chroma
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

index_seconds = {}  # 검색기별 색인 시간(초)

# 1. BM25 색인(상위 10개, 낱말은 kiwi_tokenize로)
start = time.perf_counter()
bm25 = ...
index_seconds["BM25"] = time.perf_counter() - start

# 2. Chroma 메모리 컬렉션을 비우고 KURE-v1로 문서를 임베딩해 넣기
vector_store = ...
...
start = time.perf_counter()
...
index_seconds["KURE-v1"] = time.perf_counter() - start

# 3. 의미 검색기와 하이브리드 검색기(상위 10개씩 합침)
dense = ...
hybrid = ...

# 4. 이름으로 골라 쓰도록 dict에 담기
retrievers = ...
print("BM25 색인(초):", round(index_seconds["BM25"], 3), "| KURE-v1 임베딩과 적재(초):", round(index_seconds["KURE-v1"], 3))

In [ ]:
# [자가채점] 세 검색기의 설정과 컬렉션 건수를 확인하고, 질문 하나로 세 검색기를 실행해 봅니다.
assert isinstance(bm25, BM25Retriever), "bm25에 BM25Retriever.from_documents로 만든 검색기를 담으세요. `...`를 채웠는지도 확인하세요."
assert bm25.k == 10, "bm25는 상위 10개를 돌려주도록 k=10으로 만드세요."
assert bm25.preprocess_func is kiwi_tokenize, "bm25의 preprocess_func에 kiwi_tokenize를 넘기세요."
assert len(bm25.docs) == len(drug_documents), "bm25는 drug_documents 전체로 만드세요."

assert isinstance(vector_store, Chroma), "vector_store에 Chroma 저장소를 담으세요."
assert vector_store.embeddings is embedding_model, "Chroma의 embedding_function에 embedding_model을 넘기세요."
stored_ids = vector_store.get()["ids"]
assert len(stored_ids) == len(drug_documents), f"컬렉션에 문서 {len(drug_documents)}건이 있어야 합니다(지금 {len(stored_ids)}건). 넣기 전에 reset_collection()으로 비웠는지, add_documents로 drug_documents를 넣었는지 확인하세요."
assert set(stored_ids) == set(drug_ids), "컬렉션의 ID가 원문 ID와 다릅니다. drug_documents를 그대로 add_documents에 넘기세요(Document의 id가 원문 ID입니다)."

assert dense.vectorstore is vector_store, "dense는 vector_store.as_retriever로 만드세요."
assert dense.search_kwargs == {"k": 10}, "dense의 search_kwargs는 {\"k\": 10}입니다."
assert isinstance(hybrid, EnsembleRetriever), "hybrid에 EnsembleRetriever를 담으세요."
assert len(hybrid.retrievers) == 2 and hybrid.retrievers[0] is bm25 and hybrid.retrievers[1] is dense, "hybrid의 retrievers는 [bm25, dense] 순서입니다."
assert hybrid.weights == [0.5, 0.5] and hybrid.c == 60, "hybrid는 weights=[0.5, 0.5], c=60으로 만드세요."
assert hybrid.id_key == "source_id", "hybrid의 id_key는 \"source_id\"입니다. 두 검색기가 같은 문서를 원문 ID로 알아봅니다."

assert isinstance(retrievers, dict) and list(retrievers) == ["bm25", "dense", "hybrid"], "retrievers의 키는 \"bm25\", \"dense\", \"hybrid\" 순서입니다."
assert retrievers["bm25"] is bm25 and retrievers["dense"] is dense and retrievers["hybrid"] is hybrid, "retrievers의 값은 위에서 만든 세 검색기입니다."
assert set(index_seconds) == {"BM25", "KURE-v1"}, "index_seconds의 키는 \"BM25\", \"KURE-v1\"입니다."
for name, seconds in index_seconds.items():
    assert seconds > 0, f"index_seconds[\"{name}\"]: 골격의 시간 측정 줄 사이에서 색인·적재를 실행하세요."

# 질문 하나를 세 검색기로 검색해 원문 ID가 있는 결과가 10건 이상인지
for name, retriever in retrievers.items():
    found = retriever.invoke(questions[0]["question"])
    assert len(found) >= 10, f"{name}: 결과가 {len(found)}건입니다. 상위 10개 설정을 확인하세요."
    assert found[0].metadata["source_id"] in drug_ids, f"{name}: 결과 metadata에 원문 ID(source_id)가 없습니다."
print("세 검색기 확인 완료")

### 2-2. 유형 × 전략 표

**배경**: 유형마다 어느 전략의 Recall@5가 가장 높은지 비교하려고, `search_types` 유형의 질문 36개를 세 검색기로 모두 검색해 한 표에 모읍니다(교안 02의 3-2).

**요구사항**
- **`strategies`**: 골격에 있는 전략 이름 리스트 `["bm25", "dense", "hybrid"]`(단순한 전략부터, 표의 열 순서)
- **`results`**: DataFrame. `search_types` 유형의 질문 × `strategies` 한 행씩(36 × 3 = 108행). 열은 `question_id`, `question_type`, `strategy`, `recall_at_5`(상위 5개의 Recall), `맞힌 문항`(`recall_at_5`가 0보다 크면 1, 아니면 0) 순서
- **`type_table`**: DataFrame. 행은 `search_types` 순서, 열은 `strategies` 순서, 칸은 그 유형·전략의 `recall_at_5` 평균을 소수 셋째 자리로 반올림한 값

**확인 기준**: 조건 질문도 `filter` 없이 질문 문장 그대로 검색합니다. 자가채점 셀이 행 수와 열, 표 값과, 질문 36개를 다시 검색한 Recall@5를 비교합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 질문마다 유형이 search_types에 있을 때만 세 검색기로 검색해 (질문, 전략) 한 행씩 모으고, 다 모은 뒤 피벗 표로 평균을 낸다.

세부구현:
1. questions를 돌며 question_type이 search_types에 없으면 건너뛴다.
2. strategies를 돌며 retrievers[전략]으로 질문 문장을 검색하고, 결과 Document의 metadata["source_id"]를 순서대로 모은다.
3. 모은 ID와 그 질문의 정답 ID 목록(`question["relevant_doc_ids"]`)을 recall_at_k에 넘기되 k는 5로 하고, 다섯 열을 dict 한 행으로 rows에 더한다.
4. rows로 DataFrame을 만들고 pivot_table로 유형을 행, 전략을 열, recall_at_5를 값으로 평균을 낸 뒤 소수 셋째 자리로 반올림(round)한다.
5. reindex로 행은 search_types, 열은 strategies 순서로 맞춘다.
```

</details>

In [ ]:
# [작성] 검색 질문 36개를 세 검색기로 검색해 유형 x 전략 표를 만드세요.
strategies = ["bm25", "dense", "hybrid"]  # 단순한 전략부터(점수가 같으면 앞의 전략을 고름)

# 1. 질문마다 세 검색기로 검색해 (질문, 전략) 한 행씩
rows = []
...
results = pd.DataFrame(rows)

# 2. 유형 x 전략 칸마다 평균 Recall@5
type_table = ...
display(type_table)

In [ ]:
# [자가채점] 평가 결과의 모양과 열, 표 값과, 질문마다 다시 검색한 값을 확인합니다.
expected_columns = ["question_id", "question_type", "strategy", "recall_at_5", "맞힌 문항"]
assert isinstance(results, pd.DataFrame), "results에 DataFrame을 담으세요. `...`를 채웠는지도 확인하세요."
assert list(results.columns) == expected_columns, f"results의 열은 이 순서입니다: {expected_columns}"

# (질문, 전략) 쌍이 검색 질문 x 세 전략과 한 번씩 맞는지
search_ids = set(question_table.loc[question_table["question_type"].isin(search_types), "question_id"])
pairs = set(zip(results["question_id"], results["strategy"]))
assert len(results) == len(pairs) == len(search_ids) * len(strategies), f"results는 {len(search_ids)} x {len(strategies)} = {len(search_ids) * len(strategies)}행이고 (질문, 전략) 쌍이 한 번씩이어야 합니다."
assert set(results["question_id"]) == search_ids and set(results["strategy"]) == set(strategies), "results에는 search_types 유형의 질문만, strategies의 전략마다 한 행씩 담으세요."
assert (results["맞힌 문항"] == (results["recall_at_5"] > 0).astype(int)).all(), "맞힌 문항은 recall_at_5가 0보다 크면 1, 아니면 0입니다."

# 표 모양과 값(results의 평균을 소수 셋째 자리로)
assert isinstance(type_table, pd.DataFrame), "type_table에 피벗 표(DataFrame)를 담으세요."
assert list(type_table.index) == search_types and list(type_table.columns) == strategies, "type_table의 행은 search_types, 열은 strategies 순서로 맞추세요."
recomputed = results.groupby(["question_type", "strategy"])["recall_at_5"].mean().round(3)
for (question_type, strategy), value in recomputed.items():
    assert abs(type_table.loc[question_type, strategy] - value) < 1e-9, f"{question_type} x {strategy}: 칸 값은 recall_at_5 평균을 소수 셋째 자리로 반올림한 값입니다."

# 행마다 적힌 질문·전략으로 다시 검색해 그 행의 값 확인(상위 5개로 계산했는지)
question_by_id = {question["question_id"]: question for question in questions}
for row in results.to_dict("records"):
    question = question_by_id[row["question_id"]]
    assert row["question_type"] == question["question_type"], f"{row['question_id']}: question_type은 그 질문의 유형입니다."
    found_ids = [doc.metadata["source_id"] for doc in retrievers[row["strategy"]].invoke(question["question"])]
    assert abs(row["recall_at_5"] - recall_at_k(found_ids, question["relevant_doc_ids"], 5)) < 1e-9, f"{row['question_id']} x {row['strategy']}: recall_at_5는 그 전략의 검색기로 질문 문장을 검색한 결과의 상위 5개 Recall입니다."
print("유형 x 전략 표 확인 완료")

**결과 읽기** (각 칸은 유형별 평균 Recall@5)
- 묘사 질문은 `dense`가 0.583으로 `hybrid`(0.375)와 `bm25`(0.167)보다 높습니다. 안내문과 다른 낱말로 설명한 질문이라 낱말이 겹쳐야 점수가 오르는 BM25가 약합니다.
- 이름 질문은 `dense`와 `hybrid`가 0.833으로 같고 `bm25`가 0.750입니다.
- 조건 질문은 `hybrid`가 0.431로 가장 높고 `dense`(0.250)가 가장 낮습니다. 필터 없이 질문 문장 그대로 검색한 값입니다.

## 3. 유형마다 다른 전략을 쓸 가치가 있는지 비교하세요

**배경**: 유형마다 가장 높은 전략을 쓰려면 질문 유형을 고르는 라우터가 필요하므로, 전략 하나로 모든 검색 질문을 처리할 때보다 맞힌 문항이 라우터 기준(2문항 이상)만큼 늘어나는지 확인합니다(교안 02의 3-3).

**요구사항**
- **`routed_plan`**: dict. `type_table`의 유형마다 Recall@5가 가장 높은 전략(같으면 `strategies` 앞쪽 전략)
- **`overall`**: dict. 키 `"bm25"`, `"dense"`, `"hybrid"`, `"유형별"`. 전략 셋은 `type_table`의 그 열 평균, `"유형별"`은 유형마다 가장 높은 칸의 평균이고 모두 소수 셋째 자리로 반올림. 세 유형의 질문 수가 12개씩 같아 유형 평균의 평균이 곧 전체 Recall@5입니다.
- **`hit_counts`**: dict. `overall`과 같은 네 키, 값은 int. 전략 셋은 `results`에서 그 전략의 `맞힌 문항` 합, `"유형별"`은 유형마다 `routed_plan`의 전략으로 맞힌 문항을 더한 값
- **`best_single`**: str. 전략 셋 가운데 `overall`이 가장 높은 전략(같으면 `strategies` 앞쪽)
- **`use_router`**: bool. `hit_counts["유형별"]`이 `hit_counts[best_single]`보다 2 이상 크면 `True`

**확인 기준**: `overall["유형별"]`은 전략 셋의 어느 값보다 작지 않습니다. 자가채점 셀이 다섯 값을 `type_table`과 `results`에서 다시 구해 비교합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 3-3처럼 type_table의 행마다 가장 높은 열을 고르고, 열 평균과 행 최댓값의 평균으로 전체 Recall@5를 비교한다. 맞힌 문항은 results의 맞힌 문항 열을 더한다.

세부구현:
1. DataFrame 메서드 idxmax를 행마다 적용하면(축은 열 방향) 행마다 가장 큰 값이 있는 열 이름이 앞쪽 열부터 나온다. 결과를 dict로 바꿔 routed_plan에 담는다.
2. 열마다 평균(mean)을 내 소수 셋째 자리로 반올림한 값을 dict로 바꾸고, "유형별" 키에는 행마다 최댓값(max)을 구한 뒤 그 평균을 반올림해 넣는다.
3. 전략마다 results에서 그 전략 행의 맞힌 문항을 더해 int로 담는다.
4. 유형마다 results에서 그 유형이면서 routed_plan[유형] 전략인 행의 맞힌 문항을 더해 "유형별" 값을 만든다.
5. strategies를 앞에서부터 돌며 overall이 지금까지보다 클 때만 best_single을 바꾼다.
6. 두 맞힌 문항의 차이를 2와 비교해 use_router에 담는다.
```

</details>

In [ ]:
# [작성] 유형별로 고를 때와 전략 하나로 처리할 때를 비교하고 라우터를 둘지 정하세요.
# 1. 유형마다 Recall@5가 가장 높은 전략
routed_plan = ...

# 2. 전략 하나와 유형별의 전체 Recall@5
overall = ...

# 3. 전략 하나와 유형별의 맞힌 문항
hit_counts = ...

# 4. 전략 하나 가운데 전체 Recall@5가 가장 높은 전략
best_single = ...

# 5. 라우터 기준: 유형별이 2문항 이상 더 맞히는가
use_router = ...
print("유형별 전략:", routed_plan, "| 전략 하나:", best_single, "| 라우터:", use_router)

In [ ]:
# [자가채점] 다섯 값을 type_table과 results에서 다시 구해 비교합니다.
assert isinstance(routed_plan, dict) and list(routed_plan) == search_types, "routed_plan에 search_types 유형마다 전략 하나를 담은 dict를 담으세요. `...`를 채웠는지도 확인하세요."
for question_type, chosen in routed_plan.items():
    best_value = type_table.loc[question_type].max()
    assert type_table.loc[question_type, chosen] == best_value, f"{question_type}: 그 행에서 Recall@5가 가장 높은 전략을 고르세요."
    for strategy in strategies[:strategies.index(chosen)]:
        assert type_table.loc[question_type, strategy] != best_value, f"{question_type}: 가장 높은 값이 같은 전략이 여럿이면 strategies 앞쪽 전략을 고르세요."

# overall: 소수 셋째 자리로 반올림한 값이고, 반올림 전 값과 0.0005 넘게 다르지 않음
assert isinstance(overall, dict) and set(overall) == set(strategies) | {"유형별"}, "overall의 키는 bm25, dense, hybrid, 유형별입니다."
column_means = type_table.mean()
for strategy in strategies:
    assert round(overall[strategy], 3) == overall[strategy] and abs(overall[strategy] - column_means[strategy]) <= 0.0005 + 1e-9, f"overall[\"{strategy}\"]는 type_table의 그 열 평균을 소수 셋째 자리로 반올림한 값입니다."
assert round(overall["유형별"], 3) == overall["유형별"] and abs(overall["유형별"] - type_table.max(axis=1).mean()) <= 0.0005 + 1e-9, "overall[\"유형별\"]은 유형마다 가장 높은 칸의 평균을 반올림한 값입니다."

# hit_counts: 전략별 합계 표와, 행마다 그 유형의 routed_plan 전략인지 표시한 열로 대조
assert isinstance(hit_counts, dict) and set(hit_counts) == set(overall), "hit_counts의 키는 overall과 같은 네 개입니다."
strategy_totals = results.groupby("strategy")["맞힌 문항"].sum()
for strategy in strategies:
    assert hit_counts[strategy] == strategy_totals[strategy], f"hit_counts[\"{strategy}\"]는 results에서 그 전략 행의 맞힌 문항 합입니다."
is_routed_row = results["question_type"].map(routed_plan) == results["strategy"]
assert hit_counts["유형별"] == results.loc[is_routed_row, "맞힌 문항"].sum(), "hit_counts[\"유형별\"]은 유형마다 routed_plan의 전략으로 맞힌 문항을 더한 값입니다(유형마다 맞힌 문항이 가장 많은 전략이 아님)."

# best_single: overall이 가장 높고, 그보다 앞쪽 전략은 모두 더 낮음
assert best_single in strategies, "best_single은 strategies 가운데 하나입니다."
for strategy in strategies:
    assert overall[strategy] <= overall[best_single], "best_single은 전략 셋 가운데 overall(전체 Recall@5)이 가장 높은 전략입니다."
for strategy in strategies[:strategies.index(best_single)]:
    assert overall[strategy] < overall[best_single], "overall이 같은 전략이 여럿이면 strategies 앞쪽을 고르세요."
assert isinstance(use_router, (bool, np.bool_)), "use_router에는 True 또는 False를 담으세요."
assert use_router == (hit_counts["유형별"] - hit_counts[best_single] >= 2), "use_router는 라우터 기준(유형별 맞힌 문항이 best_single보다 2 이상 많은지)입니다."
print("라우터 비교 확인 완료")

In [ ]:
# 유형 x 전략 맞힌 문항(유형마다 질문 12개)
type_hits = results.pivot_table(index="question_type", columns="strategy", values="맞힌 문항", aggfunc="sum")
display(type_hits.reindex(index=search_types, columns=strategies))

In [ ]:
# 전략 하나로 처리할 때와 유형별로 고를 때의 전체 Recall@5와 맞힌 문항(36문항)
display(pd.DataFrame({"전체 Recall@5": overall, "맞힌 문항": hit_counts}))

**결과 읽기**
- 전략 하나로 처리하면 `dense`가 0.555(23문항)로 가장 높고, `hybrid` 0.546(21문항), `bm25` 0.398(15문항) 순입니다.
- 유형별로 고른 상한은 0.616(24문항)입니다. Recall@5 차이 0.061은 조건 질문에서 `hybrid`(0.431)가 `dense`(0.250)보다 높은 데서만 나옵니다.
- 맞힌 문항으로는 조건 질문에서 1문항(`dense` 5문항, `hybrid` 6문항) 차이입니다. Recall@5 차이가 더 큰 것은 정답 문서가 여러 건인 조건 질문에서 `hybrid`가 이미 맞힌 질문의 정답을 더 많이 찾기 때문입니다.
- 라우터가 유형을 하나도 틀리지 않아도 1문항이라 라우터 기준(2문항 이상)에 못 미치므로, 검색 질문은 `dense` 하나로 처리합니다.

**작성 · ADR-002(검색 전략):** 아래 마크다운 셀을 더블클릭해 괄호 안 안내를 지우고 제목과 네 칸(상태·맥락·결정·결과)을 채웁니다. 제목은 고른 기술이 드러나게 씁니다(예: "ADR-00N: ○○로 △△ 사용"). 맥락은 정해야 할 문제·제약·후보를 2~3문장으로 쓰고 결정은 쓰지 않습니다. 결정은 "…를 쓴다."로 시작하고, 아래 `근거:`에 검토한 선택지마다 한 줄씩 버린 이유를 쓴 뒤 고른 것이 이긴 이유를 씁니다. 결과 아래에는 `얻는 것:`, `감수하는 것:`, `다시 검토할 조건:`(어떤 가정이나 수치가 바뀌면 이 결정을 다시 내려야 하는지: 바뀌는 것 -> 다시 할 일)을 씁니다. ADR은 서비스 문서처럼 교안·과제·절 번호 없이 사실을 직접 씁니다. 숫자는 자신이 실행한 2·3번 출력에서 옮깁니다.

**설명할 내용**
1. 상태: 제안·채택·거절·대체됨 가운데 하나(이 결정을 받아들였으면 `채택`)
2. 맥락: 정해야 할 것(검색 질문에 쓸 전략과 라우터를 둘지), 제약(LLM을 부르지 않는 검색, 라우터 기준, 정확도를 측정한 모델과 질문 수), 검토한 후보
3. 결정: 검색 질문에 쓸 전략(`best_single`), 라우터를 둘지(`use_router`), 관계·집계 질문의 처리를 "…를 쓴다."로 쓰고, `근거:`에 버린 후보마다 한 줄씩(전체 Recall@5와 맞힌 문항, 유형별로 고를 때 늘어나는 문항, 관계·집계를 검색으로 답하지 않는 이유) 쓴 뒤 고른 전략이 이긴 이유를 씁니다.
4. 결과: `얻는 것:`, `감수하는 것:`(고른 전략이 약한 유형과, 조건 질문을 필터 없이 측정한 점), `다시 검토할 조건:`(Self-Query 필터를 붙일 때, 평가 질문이 바뀔 때, 네 축 비교에서 다른 전략이 이길 때)

#### ADR-002: (고른 기술이 드러나는 제목)

- **상태**: (제안·채택·거절·대체됨 가운데 하나)
- **맥락**: (2~3문장: 정할 문제(검색 질문에 쓸 전략과 라우터를 둘지), 지켜야 할 조건(LLM을 부르지 않는 검색, 라우터 기준, 정확도를 측정한 모델과 질문), 검토한 후보(전략 셋과 유형별). 결정은 쓰지 않음)
- **결정**: (고른 것을 한 문장으로: …를 쓴다)
    - 근거: (후보마다 한 줄: 뺀 이유(전체 Recall@5와 맞힌 문항, 유형별로 고를 때 늘어나는 문항, 관계·집계를 검색으로 답하지 않는 이유). 마지막 줄은 고른 것이 이긴 이유)
- **결과**
    - 얻는 것: (이 결정으로 얻는 것)
    - 감수하는 것: (대신 감수하는 것)
    - 다시 검토할 조건: (바뀌는 가정이나 수치 -> 다시 할 일)

## 4. 저장 형식(정밀도)과 저장소를 고르세요

저장소는 전략이 정해진 뒤에 고릅니다. 먼저 KURE-v1 벡터를 메모리 예산 안에 담을 정밀도를 정하고, ADR-002의 전략·1번의 구조화 질의·그 정밀도가 저장소에 보낼 질의를 한 번에 처리하는 저장소만 남기기 때문입니다. 저장소마다 질의를 다시 구현하지 않고 교안 02 4-1 표처럼 기능으로 판정하고, 정밀도는 교안 02 4-3처럼 맞힌 문항과 크기로 정합니다.

### 4-1. 저장 형식(정밀도) 정하기

벡터 값 하나를 더 적은 비트로 저장하면(교안 02 4-3) 메모리를 아끼는 대신 순위가 흔들려 맞힌 문항이 줄 수 있습니다. 2번에서 Chroma에 넣은 KURE-v1 문서 벡터를 세 정밀도로 낮춰 검색 질문 36개로 맞힌 문항을 세고, 5만 건 벡터 크기를 메모리 예산 60MB와 견줍니다. 정밀도를 바꾼 벡터는 Chroma에 넣을 수 없어 NumPy로 모든 문서와 비교합니다(정확 검색).

| 제공 코드 | 입력과 반환값 |
|---|---|
| `PRECISIONS`, `ASSUMED_DOCS`, `BUDGET_MB` | 정밀도 이름 목록 `["float32", "float16", "binary"]`, 가정한 품목 수 50,000, 메모리 예산 60 |
| `lower_precision(vectors, precision)` | 문서나 질문 벡터를 `precision`으로 낮춘 값을 담은 float32 배열(거리 계산은 float32로) |
| `storage_mb(n_docs, dimension, precision)` | 벡터만 센 저장 크기(MB, 100만 바이트), 소수 첫째 자리 |
| `doc_vectors` | Chroma에서 꺼낸 KURE-v1 문서 벡터(394 × 1024, `drug_ids` 순서) |
| `eval_positions`, `question_vectors` | 검색 질문 36개가 `questions`에서 몇 번째인지 담은 목록과, 그 순서의 KURE-v1 질문 벡터(36 × 1024) |
| `top5_ids(question_vector, doc_vectors)` | 질문 벡터 하나를 모든 문서 벡터와 내적해 큰 순서로 원문 ID 5개 |
| `count_found(found_id_lists)` | 검색 질문마다(`eval_positions` 순서) 정답이 상위 5개 안에 하나라도 있으면 맞힌 문항으로 세어 반환(과제 01과 같은 기준) |

In [ ]:
# 정밀도별 값 하나의 바이트(binary는 1비트 = 1/8바이트)
BYTES_PER_VALUE = {"float32": 4, "float16": 2, "binary": 1 / 8}
PRECISIONS = ["float32", "float16", "binary"]

# 서비스 가정: 품목 5만 건, 검색 서버가 벡터에 줄 수 있는 메모리 60MB
ASSUMED_DOCS = 50_000
BUDGET_MB = 60


def lower_precision(vectors, precision):
    """벡터(문서 또는 질문)를 precision 정밀도로 낮춘 값을 담은 float32 배열을 반환합니다."""
    vectors = np.asarray(vectors, dtype=np.float32)
    # float16: 값 하나를 2바이트 정밀도로 반올림(거리 계산은 float32로 하므로 다시 float32 배열로 둠)
    if precision == "float16":
        return vectors.astype(np.float16).astype(np.float32)
    # binary: 부호만 남겨 1과 -1로(저장은 1비트, 1과 -1의 내적은 해밍 거리와 순위가 같음)
    if precision == "binary":
        return (vectors > 0).astype(np.float32) * 2 - 1
    return vectors


def storage_mb(n_docs, dimension, precision):
    """벡터만 센 저장 크기(MB, 100만 바이트)를 소수 첫째 자리로 반환합니다."""
    return round(n_docs * dimension * BYTES_PER_VALUE[precision] / 1e6, 1)


def top5_ids(question_vector, doc_vectors):
    """질문 하나를 모든 문서와 내적해(길이 1이면 코사인 유사도) 큰 순서로 원문 ID 5개를 반환합니다."""
    scores = np.asarray(doc_vectors) @ np.asarray(question_vector)
    top_positions = np.argsort(-scores, kind="stable")[:5]  # 동점은 문서 순서를 지켜 실행마다 같은 결과
    return [drug_ids[position] for position in top_positions]


# 2번에서 Chroma에 넣은 KURE-v1 문서 벡터를 drug_ids 순서로 꺼내기(다시 임베딩하지 않음)
stored = vector_store.get(include=["embeddings"])
vector_by_id = dict(zip(stored["ids"], stored["embeddings"]))
doc_vectors = np.array([vector_by_id[doc_id] for doc_id in drug_ids], dtype=np.float32)

# 검색 질문 36개의 위치와 KURE-v1 질문 벡터
eval_positions = []
for index, question in enumerate(questions):
    if question["question_type"] in search_types:
        eval_positions.append(index)
question_vectors = np.array(embedding_model.embed_documents([questions[index]["question"] for index in eval_positions]), dtype=np.float32)


def count_found(found_id_lists):
    """검색 질문마다 정답이 상위 5개 안에 하나라도 있으면 맞힌 문항으로 세어 반환합니다."""
    found = 0
    for index, found_ids in zip(eval_positions, found_id_lists, strict=True):
        found += int(recall_at_k(found_ids, questions[index]["relevant_doc_ids"], 5) > 0)
    return found


print("문서 벡터:", doc_vectors.shape, "| 질문 벡터:", question_vectors.shape)

**배경**: 메모리 예산을 넘는 정밀도는 쓸 수 없고, 예산 안에서도 맞힌 문항이 같은 수준이면 작은 쪽이 낫기 때문에 정밀도마다 두 값을 한 표에 모아 저장 형식 기준으로 하나를 고릅니다.

**요구사항**
- **`format_rows`**: list[dict], `PRECISIONS` 순서로 세 행. 행은 `precision`, `맞힌 문항`, `storage_mb_50k` 세 키입니다. `맞힌 문항`은 문서 벡터(`doc_vectors`)와 질문 벡터(`question_vectors`)를 **같은 정밀도로 함께** 낮춰 질문마다 `top5_ids`로 찾은 결과를 `count_found`로 센 값이고(저장소는 질문도 저장 형식에 맞춰 바꿔 비교함), `storage_mb_50k`는 `storage_mb(ASSUMED_DOCS, 문서 벡터의 열 수인 차원, 그 정밀도)`입니다.
- **`chosen_precision`**: str. `storage_mb_50k`가 `BUDGET_MB` 이하인 행만 남기고(같으면 통과), 그 가운데 `맞힌 문항`이 가장 많은 행과 1문항 이내인 행에서 `storage_mb_50k`가 가장 작은 행의 `precision`

**확인 기준**: 표에서 float32와 float16의 5만 건 크기는 각각 204.8MB, 102.4MB입니다. 자가채점 셀이 binary 행을 다시 검색해 문서·질문을 함께 낮췄는지 확인합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 정밀도마다 문서·질문 벡터를 같은 함수로 낮추고, 낮춘 질문 벡터 하나씩 낮춘 문서 벡터에서 상위 5개를 찾아 센다. 고르기는 예산 거르기, 1문항 이내 거르기, 가장 작은 크기 고르기 세 단계다.

세부구현:
1. PRECISIONS를 돌며 lower_precision으로 doc_vectors와 question_vectors를 낮춘다.
2. 낮춘 질문 벡터마다 top5_ids에 그 질문 벡터와 낮춘 문서 벡터를 넘겨 결과를 모으고, 모은 목록을 count_found에 넘긴다.
3. precision, 맞힌 문항, storage_mb_50k 세 키의 dict를 format_rows에 더한다.
4. 예산 이하 행만 리스트로 모으고, 그 안에서 가장 많은 맞힌 문항을 구한다.
5. 그 값과 1문항 이내인 행만 다시 모은 뒤, 반복문으로 storage_mb_50k가 가장 작은 행을 찾아 precision을 꺼낸다.
```

</details>

In [ ]:
# [작성] 정밀도마다 맞힌 문항과 5만 건 크기를 세고, 저장 형식 기준으로 하나를 고르세요.
format_rows = []
for precision in PRECISIONS:
    # 1. 문서 벡터와 질문 벡터를 같은 정밀도로 낮추기
    ...

    # 2. 질문마다 정확 검색해 맞힌 문항 세기
    ...

    # 3. 정밀도, 맞힌 문항, 5만 건 크기를 한 행으로
    ...
display(pd.DataFrame(format_rows))

# 4. 예산 안 -> 1문항 이내 -> 크기가 가장 작은 정밀도
chosen_precision = ...
print("고른 정밀도:", chosen_precision)

In [ ]:
# [자가채점] 행의 모양·크기, 정밀도마다 다시 센 맞힌 문항, 고른 정밀도가 기준을 따랐는지 확인합니다.
assert isinstance(format_rows, list) and len(format_rows) == len(PRECISIONS), "format_rows에 PRECISIONS 순서로 세 행을 담으세요. `...`를 채웠는지도 확인하세요."
dimension = doc_vectors.shape[1]

# 행마다 그 정밀도로 문서·질문 벡터를 함께 낮춰 맞힌 문항을 다시 세어 비교(이 PC의 벡터로 세므로 PC마다 값이 달라도 됨)
for precision, row in zip(PRECISIONS, format_rows):
    assert set(row) == {"precision", "맞힌 문항", "storage_mb_50k"}, "행은 precision, 맞힌 문항, storage_mb_50k 세 키의 dict입니다."
    assert row["precision"] == precision, "format_rows는 PRECISIONS 순서입니다."
    assert row["storage_mb_50k"] == storage_mb(ASSUMED_DOCS, dimension, precision), f"{precision}: storage_mb에 ASSUMED_DOCS, 문서 벡터의 차원, 그 행의 정밀도를 넘기세요."
    check_docs = lower_precision(doc_vectors, precision)
    rerun_found = count_found([top5_ids(vector, check_docs) for vector in lower_precision(question_vectors, precision)])
    assert row["맞힌 문항"] == rerun_found, f"{precision}: 맞힌 문항이 {row['맞힌 문항']}인데 다시 세면 {rerun_found}입니다. doc_vectors와 question_vectors를 같은 정밀도로 함께 낮췄는지, 질문마다 top5_ids로 찾은 결과를 count_found로 셌는지 확인하세요."

# 고른 정밀도: 예산 안이고, 예산 안의 어느 행보다도 2문항 이상 적지 않으며, 그보다 작은 예산 안 행은 같은 수준이 아님
chosen_rows = [row for row in format_rows if row["precision"] == chosen_precision]
assert len(chosen_rows) == 1, "chosen_precision은 PRECISIONS 가운데 하나입니다."
chosen_row = chosen_rows[0]
assert chosen_row["storage_mb_50k"] <= BUDGET_MB, "5만 건 크기가 메모리 예산(BUDGET_MB)을 넘는 정밀도는 고를 수 없습니다."
in_budget_rows = [row for row in format_rows if row["storage_mb_50k"] <= BUDGET_MB]
for row in in_budget_rows:
    assert row["맞힌 문항"] - chosen_row["맞힌 문항"] <= 1, "고른 정밀도는 예산 안에서 맞힌 문항이 가장 많은 행과 1문항 이내여야 합니다."
    if row["storage_mb_50k"] < chosen_row["storage_mb_50k"]:
        is_same_level = True
        for other in in_budget_rows:
            if other["맞힌 문항"] - row["맞힌 문항"] > 1:
                is_same_level = False
        assert not is_same_level, "같은 수준(1문항 이내)인 정밀도가 여럿이면 5만 건 크기가 가장 작은 것을 고르세요."
print("저장 형식 확인 완료:", chosen_precision)

### 4-2. 서비스가 보낼 질의로 저장소 고르기

| 저장소 | 질의 한 번으로 하는 일 | 하지 못하는 일 |
|---|---|---|
| 전용 벡터 DB(Chroma) | `query`로 벡터 상위 k, `where`로 메타데이터 조건 안 상위 k, `get(where=...)`로 조건에 맞는 항목 꺼내기 | 항목끼리 잇는 관계를 담지 못함. 조건에 맞는 개수만 돌려주는 질의가 없음(꺼낸 항목을 파이썬에서 셈). 순위를 매기는 키워드 검색이 없음. 로컬 HNSW 설정은 거리·이웃 수·탐색 폭뿐이라 벡터를 1비트로 줄여 담는 양자화가 없음 |
| pgvector | `ORDER BY embedding <=> 질문 LIMIT k` 상위 k, `WHERE` 조건, 여러 테이블 `JOIN`, `count(*)`, `binary_quantize`로 만든 `bit` 식 인덱스로 후보를 뽑아 원래 벡터 `<=>`로 다시 정렬하는 서브쿼리 | 한국어 키워드 검색은 형태소 분석을 밖에서 해서 전문 검색에 넣어야 함 |
| Neo4j | 벡터 인덱스 `SEARCH` 상위 k, 필터용 인덱스의 `SEARCH` 안 `WHERE`, 경로 패턴 `(a)-[:WARNS]->(i)<-[:CONTAINS]-(b)`, `count()`, 전문 인덱스 키워드 검색, 벡터 인덱스 설정 `vector.quantization.type: 'binary'`(인덱스가 원래 벡터로 다시 점수를 매김) | 없음 |

- 성분 관계는 pgvector에서는 성분 테이블, Neo4j에서는 `CONTAINS`·`WARNS` 관계로 저장소 안에 둡니다. Chroma는 담을 자리가 없어 성분 관계 표를 파이썬에서 따라간 뒤 얻은 ID를 `where`에 넣어야 합니다.
- 관계·집계 질의는 Text2Cypher·Text-to-SQL이 질문마다 만들어 보냅니다. 여기서는 그 질의를 저장소가 한 번에 받아 처리하는지만 봅니다.

| 판정 값 | 뜻 |
|---|---|
| `한 질의` | 저장소에 질의를 한 번 보내 필요한 답(순위 목록이나 개수)을 그대로 받음 |
| `코드로 우회` | 저장소 기능을 쓰되, 밖(파이썬, 따로 둔 데이터)에서 만든 값을 넣어 질의하거나 받은 결과를 파이썬에서 다시 거르거나 셈 |
| `불가` | 저장소에 그 질의의 기능이 아예 없어 다른 도구가 대신해야 함 |

In [ ]:
# 필요한 곳(검색 전략, 조건 필터, 구조화 질의 유형, 저장 형식)마다 저장소에 보낼 질의
query_by_need = {
    "bm25": "키워드 상위 k",
    "dense": "벡터 상위 k",
    "hybrid": "키워드 + 벡터 순위 합치기",
    "조건 필터": "제형 조건 안 벡터 상위 k",
    "관계": "성분을 거쳐 다른 제품 찾기",
    "집계": "제형·효능군 조건 세기",
    "float32": "float32 벡터 인덱스",
    "float16": "float16 벡터 인덱스",
    "binary": "binary 인덱스 + 원래 벡터로 재점수",
}

# 비교할 저장소와 판정 값
stores = ["Chroma", "pgvector", "Neo4j"]
support_values = ["한 질의", "코드로 우회", "불가"]

# (가정) 본사가 이미 운영하는 DB
operated_db = "Neo4j"

**배경**: 저장소는 "어느 제품이 빠른가"가 아니라 서비스가 보낼 질의를 처리하는가로 고르므로, 앞에서 정한 전략·구조화 질의·정밀도가 요구하는 질의를 모아 저장소마다 판정합니다(교안 02의 4-1).

**요구사항**
- **`required_queries`**: list[str]. `query_by_need`의 값으로 아래 순서대로 담습니다.
    1. 3번에서 고른 전략(`best_single`)의 질의
    2. 조건 질문에 Self-Query로 필터를 붙일 때의 질의(`"조건 필터"`)
    3. `answer_path`의 값이 `"구조화 질의"`인 유형마다 그 유형의 질의(1번 표의 유형 순서)
    4. 4-1에서 고른 정밀도(`chosen_precision`)의 질의
- **`store_support`**: DataFrame. 행은 `required_queries`, 열은 `stores`(같은 순서), 칸은 위 기능 표와 판정 값의 뜻에 따라 `support_values` 가운데 하나

**확인 기준**: 판정 표는 5 × 3입니다. 자가채점 셀이 칸마다 판정 값의 지문을 비교합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 질의 목록은 앞에서 정한 변수 넷을 query_by_need의 키로 써서 순서대로 모은다. 판정은 저장소마다 질의 하나씩, 기능 표의 '하는 일'과 '하지 못하는 일'에서 찾는다.

세부구현:
1. query_by_need에서 3번에서 고른 전략 이름과 "조건 필터"를 키로 차례로 꺼내 리스트를 시작한다.
2. 1번 표의 유형 순서로 돌며 answer_path가 "구조화 질의"인 유형의 query_by_need 값을 더한다.
3. 끝에 4-1에서 고른 정밀도 이름을 키로 꺼낸 값을 더한다.
4. 저장소 이름을 키로, {질의: 판정 값} dict를 값으로 하는 dict를 만들어 pd.DataFrame에 넘긴다.
5. .loc[required_queries]로 행 순서를 맞추고 열 순서가 stores와 같은지 본다.
```

</details>

In [ ]:
# [작성] 필요한 질의를 모으고, 저장소마다 한 질의로 처리하는지 판정하세요.
# 1. 전략, 조건 필터, 구조화 질의 유형, 정밀도 순서로 질의 모으기
required_queries = ...

# 2. 행은 required_queries, 열은 stores, 칸은 support_values 가운데 하나
store_support = ...
display(store_support)

In [ ]:
# [자가채점] 질의 목록이 앞의 결정에서 나왔는지, 판정 표의 모양과 칸 값을 확인합니다.
assert isinstance(required_queries, list), "required_queries에 질의 이름 리스트를 담으세요. `...`를 채웠는지도 확인하세요."
# 질의 목록 전체의 지문(3번에서 고른 best_single마다 하나, "|"로 이은 문자열의 sha256 앞 12글자만 둠)
required_fingerprints = {"bm25": "221a76c3a4cd", "dense": "d3f20b48cea6", "hybrid": "6d4225e233be"}
assert best_single in required_fingerprints, "best_single이 strategies 가운데 하나인지 3번을 다시 확인하세요."
digest = hashlib.sha256("|".join(required_queries).encode("utf-8")).hexdigest()[:12]
assert digest == required_fingerprints[best_single], "required_queries는 3번의 best_single, \"조건 필터\", 구조화 질의 유형(1번 표 순서), 4-1의 chosen_precision을 query_by_need로 바꿔 이 순서로 담습니다."

assert isinstance(store_support, pd.DataFrame), "store_support에 DataFrame을 담으세요."
assert list(store_support.index) == required_queries, "행은 required_queries 순서로 맞추세요."
assert list(store_support.columns) == stores, "열은 stores 순서(Chroma, pgvector, Neo4j)로 맞추세요."

# 칸마다 정답 값의 지문(정답을 그대로 적지 않으려고 "질의|저장소|값"의 sha256 앞 12글자만 둠)
support_fingerprints = {
    ("벡터 상위 k", "Chroma"): "0e67892157a4",
    ("제형 조건 안 벡터 상위 k", "Chroma"): "c437d59991ca",
    ("성분을 거쳐 다른 제품 찾기", "Chroma"): "44ed45f358b4",
    ("제형·효능군 조건 세기", "Chroma"): "d2c062c1df28",
    ("binary 인덱스 + 원래 벡터로 재점수", "Chroma"): "59ba0f7f7318",
    ("키워드 상위 k", "Chroma"): "feb5db8d5bde",
    ("키워드 + 벡터 순위 합치기", "Chroma"): "921a36afba19",
    ("벡터 상위 k", "pgvector"): "8e3b362e6a80",
    ("제형 조건 안 벡터 상위 k", "pgvector"): "a43ba87ab06e",
    ("성분을 거쳐 다른 제품 찾기", "pgvector"): "9d36ac5864b6",
    ("제형·효능군 조건 세기", "pgvector"): "a2c578855c5a",
    ("binary 인덱스 + 원래 벡터로 재점수", "pgvector"): "06dbaa972f88",
    ("키워드 상위 k", "pgvector"): "e594cc3f5361",
    ("키워드 + 벡터 순위 합치기", "pgvector"): "b1f7e6181a1a",
    ("벡터 상위 k", "Neo4j"): "a7c21796ab36",
    ("제형 조건 안 벡터 상위 k", "Neo4j"): "3375ce7cb39a",
    ("성분을 거쳐 다른 제품 찾기", "Neo4j"): "562fd098509d",
    ("제형·효능군 조건 세기", "Neo4j"): "f8451c2dd933",
    ("binary 인덱스 + 원래 벡터로 재점수", "Neo4j"): "147804b0a175",
    ("키워드 상위 k", "Neo4j"): "ee116e6f1316",
    ("키워드 + 벡터 순위 합치기", "Neo4j"): "211fa7a5749f"
}
for query_name in required_queries:
    for store in stores:
        value = store_support.loc[query_name, store]
        assert value in support_values, f"{query_name} x {store}: 칸에는 {support_values} 가운데 하나를 넣으세요."
        assert (query_name, store) in support_fingerprints, f"{query_name}: 행 이름은 query_by_need의 질의 이름을 그대로 쓰세요."
        digest = hashlib.sha256(f"{query_name}|{store}|{value}".encode("utf-8")).hexdigest()[:12]
        assert digest == support_fingerprints[(query_name, store)], f"{query_name} x {store}: 기능 표와 판정 값의 뜻을 다시 보세요. 저장소 안의 질의 한 번으로 답이 나오는지, 밖에서 만든 값이 필요한지, 기능이 아예 없는지 따져 보세요."
print("저장소 판정 확인 완료")

**배경**: 판정 표에서 필요한 질의를 모두 한 질의로 처리하는 저장소가 여럿이면, 새 DB를 들이지 않는 쪽이 운영 부담이 작아 선정 기준대로 이미 운영하는 DB를 고릅니다.

**요구사항**
- **`single_query_stores`**: list[str]. `stores` 가운데 `store_support`의 그 열이 모두 `"한 질의"`인 저장소(`stores` 순서)
- **`chosen_store`**: str. `single_query_stores`가 둘 이상이고 그 안에 `operated_db`가 있으면 `operated_db`, 아니면 `single_query_stores`의 첫 저장소

**확인 기준**: `chosen_store`는 `single_query_stores` 가운데 하나입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 열마다 모든 칸이 "한 질의"인지 확인해 저장소를 모으고, 운영 기준으로 하나를 고른다.

세부구현:
1. stores를 돌며 store_support의 그 저장소 열을 "한 질의"와 비교하고, 비교 결과의 all 메서드가 참인 저장소만 리스트에 더한다.
2. 그 리스트의 첫 저장소를 chosen_store로 두고, 둘 이상이면서 operated_db가 들어 있으면 operated_db로 바꾼다.
```

</details>

In [ ]:
# [작성] 필요한 질의를 모두 한 질의로 처리하는 저장소 가운데 하나를 고르세요.
# 1. 열이 모두 '한 질의'인 저장소
single_query_stores = ...
print("필요한 질의를 모두 한 질의로:", single_query_stores)

# 2. 둘 이상이면 이미 운영하는 DB
chosen_store = ...
print("고른 저장소:", chosen_store)

In [ ]:
# [자가채점] 고른 저장소가 판정 표와 운영 기준을 따랐는지 확인합니다.
assert isinstance(single_query_stores, list), "single_query_stores에 저장소 이름 리스트를 담으세요. `...`를 채웠는지도 확인하세요."
for store in stores:
    column_values = set(store_support[store])
    if store in single_query_stores:
        assert column_values == {"한 질의"}, f"{store}: 열에 '한 질의'가 아닌 칸이 있어 single_query_stores에 넣을 수 없습니다."
    else:
        assert column_values != {"한 질의"}, f"{store}: 열의 칸이 모두 '한 질의'라 single_query_stores에 넣어야 합니다."
assert single_query_stores == [store for store in stores if store in single_query_stores], "single_query_stores는 stores 순서로 한 번씩 담으세요."
assert chosen_store in single_query_stores, "chosen_store는 single_query_stores 가운데 하나입니다."
if len(single_query_stores) > 1 and operated_db in single_query_stores:
    assert chosen_store == operated_db, "후보가 둘 이상이면 이미 운영하는 DB(operated_db)를 고르세요."
print("저장소 결정 확인 완료:", chosen_store)

**결과 읽기**
- 정밀도 표에서 예산 60MB 안에 드는 것은 binary(6.4MB)뿐이고, 1단계 binary는 16문항으로 float32·float16(23문항)보다 7문항 적습니다.
- 판정 표에서 Chroma는 성분 관계와 조건 세기를 `코드로 우회`하고 양자화가 `불가`라 빠지고, 다섯 질의를 모두 한 질의로 처리하는 pgvector와 Neo4j 가운데 이미 운영하는 Neo4j가 남습니다.
- 판정 표는 Neo4j가 binary 인덱스를 한 질의로 만든다는 것만 알려 줍니다. 교안 02 4-3의 "원본 + 낮춘 사본" 방식처럼 인덱스가 원래 벡터로 후보를 다시 점수 매겨 1단계 binary가 잃은 문항을 얼마나 되찾는지는 이 과제에서 측정하지 않으므로, ADR-003의 다시 검토할 조건에 남깁니다.

**작성 · ADR-003(저장 형식과 벡터 저장소):** 아래 마크다운 셀을 더블클릭해 괄호 안 안내를 지우고 제목과 네 칸(상태·맥락·결정·결과)을 채웁니다. 제목은 고른 기술이 드러나게 씁니다(예: "ADR-00N: ○○로 △△ 사용"). 맥락은 정해야 할 문제·제약·후보를 2~3문장으로 쓰고 결정은 쓰지 않습니다. 결정은 "…를 쓴다."로 시작하고, 아래 `근거:`에 검토한 선택지마다 한 줄씩 버린 이유를 쓴 뒤 고른 것이 이긴 이유를 씁니다. 결과 아래에는 `얻는 것:`, `감수하는 것:`, `다시 검토할 조건:`(어떤 가정이나 수치가 바뀌면 이 결정을 다시 내려야 하는지: 바뀌는 것 -> 다시 할 일)을 씁니다. ADR은 서비스 문서처럼 교안·과제·절 번호 없이 사실을 직접 씁니다. 숫자는 자신이 실행한 4번 출력에서 옮깁니다.

**설명할 내용**
1. 상태: 제안·채택·거절·대체됨 가운데 하나(이 결정을 받아들였으면 `채택`)
2. 맥락: 정해야 할 것(저장 형식과 저장소), 메모리 예산, 필요한 질의(ADR-002의 전략, 구조화 질의, 고른 정밀도에서 나옴)와 운영 중인 DB, 검토한 정밀도와 저장소
3. 결정: 저장 형식(`chosen_precision`)과 저장소(`chosen_store`)를 "…를 쓴다."로 쓰고, `근거:`에 버린 정밀도·저장소마다 한 줄씩(`format_rows`의 맞힌 문항과 크기, 판정 표, 운영 기준) 쓴 뒤 고른 것이 이긴 이유를 씁니다.
4. 결과: `얻는 것:`, `감수하는 것:`(원래 벡터를 함께 두는 점, 측정하지 않은 재점수 정확도), `다시 검토할 조건:`(재점수 정확도를 측정할 때, 메모리 예산이나 품목 수가 바뀔 때, Text2Cypher를 붙일 때, Neo4j 버전이나 운영 DB가 바뀔 때)

#### ADR-003: (고른 기술이 드러나는 제목)

- **상태**: (제안·채택·거절·대체됨 가운데 하나)
- **맥락**: (2~3문장: 정할 문제(저장 형식과 벡터 저장소), 지켜야 할 조건(메모리 예산, 필요한 질의, 운영 중인 DB), 검토한 후보(정밀도, 저장소). 결정은 쓰지 않음)
- **결정**: (고른 것을 한 문장으로: …를 쓴다)
    - 근거: (후보마다 한 줄: 뺀 이유(정밀도별 맞힌 문항과 크기, 판정 값, 운영 기준). 마지막 줄은 고른 것이 이긴 이유)
- **결과**
    - 얻는 것: (이 결정으로 얻는 것)
    - 감수하는 것: (대신 감수하는 것)
    - 다시 검토할 조건: (바뀌는 가정이나 수치 -> 다시 할 일)

## 5. 후보표를 만드세요

전략·저장소를 정했으니, 교안 02의 6-1처럼 검색 전략 후보 셋을 네 축으로 한 표에 놓아 최종 구성을 다시 확인합니다. 임베딩 모델은 KURE-v1 하나라 후보는 검색 전략만 다릅니다.

| 후보 | 검색 전략 | 쓰는 검색기 |
|---|---|---|
| A | `dense` | 벡터 인덱스(KURE-v1) |
| B | `hybrid` | 벡터 인덱스(KURE-v1) + BM25 색인 |
| C | `bm25` | BM25 색인 |

**네 축 계산 방법** (`build_candidates`가 계산합니다)
- 정확도(`accuracy`): 3번 `overall`의 그 전략 값(검색 질문 36개의 전체 Recall@5)
- 지연(`latency_ms`): 운영 장비(CPU)의 가정 시간입니다. 벡터 인덱스를 쓰면 KURE-v1 질문 임베딩 60ms, BM25 색인을 쓰면 낱말 점수 계산 1ms를 더하고, 5만 건 검색 30ms를 더합니다. 이 PC에서 측정한 검색 시간은 운영 장비를 대표하지 못해 지연에는 쓰지 않습니다.
- 비용(`cost_usd`): 월 운영비 = 색인 1회 ÷ 12개월 + 월 질의. 색인 1회는 2번에서 측정한 색인 시간(`index_seconds`)을 5만 건으로 늘려 같은 장비 가격으로 환산한 값입니다. 이 PC의 시간을 그대로 쓰지만 월 운영비에서 차지하는 몫이 아주 작아 순위에 영향을 주지 않습니다. 월 질의는 질문을 임베딩하는 전략만 질문 임베딩 장비를 한 달 내내 켜 둔 비용(시간당 가격 × 730시간)이고, BM25만 쓰는 전략은 0입니다(후보 C의 비용은 색인 비용뿐).
- 저장(`storage_gb`): 교안 02 6-1과 같은 식입니다. 원래 벡터(5만 건 × 1024차원 × 4바이트)에 교안 02 4-2에서 측정한 배수 3.0(원래 벡터 + HNSW 인덱스)을 곱하고, BM25 색인(문서당 고유 낱말 수 × 12바이트)을 쓰는 전략이면 더합니다. 교안에서는 이 값이 검색에 쓰는 메모리였지만, 이 과제는 binary 인덱스라 메모리에는 binary 사본(4-1의 60MB 예산)만 올리고 원래 벡터는 재점수용으로 함께 둡니다. 그래서 이 축은 메모리가 아니라 검색에 두는 저장 공간 전체입니다.

시나리오(7번)에서 품목 수와 장비 가격을 바꿔 다시 쓰도록 후보표를 함수로 만들었습니다.

In [ ]:
# (가정) 운영 장비(CPU)에서의 시간(ms): KURE-v1 질문 1건 임베딩, BM25 낱말 점수 계산, 5만 건 검색 한 번
assumed_embed_ms = 60.0
assumed_bm25_ms = 1.0
assumed_search_ms = 30.0

# (가정) 질문 임베딩 장비 1시간 가격과 한 달 켜 두는 시간
machine_usd_per_hour = 0.35
machine_hours_per_month = 730

# 원래 벡터 + HNSW 인덱스는 원래 벡터의 3.0배(교안 02 4-2에서 측정), (가정) BM25 낱말 하나 12바이트, 1년에 한 번 재색인
vector_index_factor = 3.0
bm25_bytes_per_term = 12
reindex_months = 12
unique_terms_per_doc = float(np.mean([len(set(kiwi_tokenize(document.page_content))) for document in drug_documents]))  # 문서당 고유 낱말 수(측정)


def monthly_cost(index_seconds_per_doc, n_docs, machine_usd_per_hour, needs_machine):
    """색인 1회(index_usd)와 월 질의(monthly_query_usd) 비용(달러)을 dict로 반환합니다."""
    # 색인 1회: 문서당 초 x 문서 수를 시간으로 바꿔 장비 가격을 곱함
    index_cost = index_seconds_per_doc * n_docs / 3600 * machine_usd_per_hour

    # 월 질의: 질문을 임베딩하는 전략은 질문 임베딩 장비를 한 달 내내 켜 둠(BM25는 검색 서버에서 바로 계산)
    query_cost = 0.0
    if needs_machine:
        query_cost = machine_usd_per_hour * machine_hours_per_month
    return {"index_usd": round(index_cost, 3), "monthly_query_usd": round(query_cost, 3)}


def build_candidates(uses_by_strategy, accuracy_by_strategy, n_docs, machine_usd_per_hour):
    """품목 수가 n_docs, 장비 가격이 machine_usd_per_hour일 때 후보 A~C의 네 축을 담은 후보표를 반환합니다."""
    rows = []
    for name, strategy in [("A", "dense"), ("B", "hybrid"), ("C", "bm25")]:
        uses_vectors = uses_by_strategy[strategy]["vectors"]  # 벡터 인덱스(KURE-v1)를 쓰는가
        uses_bm25 = uses_by_strategy[strategy]["bm25"]  # BM25 색인을 쓰는가

        # 지연: 쓰는 검색기만큼 가정 시간을 더하고 검색 시간을 더함
        latency_ms = uses_vectors * assumed_embed_ms + uses_bm25 * assumed_bm25_ms + assumed_search_ms

        # 비용: 쓰는 검색기만큼 색인 시간을 더하고, 질문을 임베딩하는 전략만 장비를 늘 켜 둠
        index_seconds_per_doc = (uses_vectors * index_seconds["KURE-v1"] + uses_bm25 * index_seconds["BM25"]) / len(drug_documents)
        cost = monthly_cost(index_seconds_per_doc, n_docs, machine_usd_per_hour, uses_vectors)

        # 저장: 원래 벡터 x 인덱스 배율 + BM25 색인
        storage_gb = (uses_vectors * n_docs * 1024 * 4 * vector_index_factor
                      + uses_bm25 * n_docs * unique_terms_per_doc * bm25_bytes_per_term) / 1e9

        rows.append({"candidate": name, "strategy": strategy, "accuracy": accuracy_by_strategy[strategy],
                     "latency_ms": round(latency_ms, 3),
                     "cost_usd": round(cost["index_usd"] / reindex_months + cost["monthly_query_usd"], 3),
                     "storage_gb": round(storage_gb, 3)})
    return pd.DataFrame(rows)

**배경**: 후보마다 지연·비용·저장 공간이 달라지는 까닭은 그 전략이 실행하는 검색기가 다르기 때문이므로, 전략마다 쓰는 검색기와 측정한 정확도를 `build_candidates`에 넘겨 후보표를 만듭니다.

**요구사항**
- **`uses_by_strategy`**: dict. 키는 `strategies`의 세 전략, 값은 `{"vectors": bool, "bm25": bool}`(그 전략이 벡터 인덱스를 쓰는가, BM25 색인을 쓰는가). 2-1에서 그 전략의 검색기를 무엇으로 만들었는지 보고 정합니다.
- **`accuracy_by_strategy`**: dict. 키는 `strategies`의 세 전략, 값은 `overall`의 그 전략 값
- **`candidates`**: DataFrame(3행). `build_candidates(uses_by_strategy, accuracy_by_strategy, ASSUMED_DOCS, machine_usd_per_hour)`의 결과. 열은 `candidate`, `strategy`, `accuracy`, `latency_ms`, `cost_usd`, `storage_gb`

**확인 기준**: 후보 B(`hybrid`)는 A(`dense`)의 지연·비용·저장 공간에 BM25 몫만큼 더한 값입니다. 자가채점 셀이 `uses_by_strategy`를 2-1의 검색기 객체와 대조합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 전략마다 retrievers의 검색기가 dense(벡터 인덱스)를 쓰는지, bm25(BM25 색인)를 쓰는지 따진다. hybrid는 두 검색기를 합친 것이다.

세부구현:
1. strategies를 돌며 그 전략의 두 bool을 담은 dict를 uses_by_strategy[전략]에 넣는다.
2. strategies를 돌며 overall[전략]을 accuracy_by_strategy[전략]에 넣는다.
3. 두 dict와 품목 수 ASSUMED_DOCS, 장비 가격 machine_usd_per_hour를 build_candidates에 넘긴다.
```

</details>

In [ ]:
# [작성] 전략마다 쓰는 검색기와 정확도를 정리해 후보표를 만드세요.
# 1. 전략마다 벡터 인덱스와 BM25 색인을 쓰는지
uses_by_strategy = ...

# 2. 전략마다 3번의 전체 Recall@5
accuracy_by_strategy = ...

# 3. 기본 시나리오(품목 5만 건, 장비 시간당 $0.35)의 후보표
candidates = ...
display(candidates)

In [ ]:
# [자가채점] 전략마다 쓰는 검색기를 2-1의 검색기 객체와 대조하고, 후보표가 기본 시나리오로 만든 표인지 확인합니다.
assert isinstance(uses_by_strategy, dict) and set(uses_by_strategy) == set(strategies), "uses_by_strategy의 키는 bm25, dense, hybrid입니다. `...`를 채웠는지도 확인하세요."
for strategy, retriever in retrievers.items():
    uses = uses_by_strategy[strategy]
    assert isinstance(uses, dict) and set(uses) == {"vectors", "bm25"}, f"uses_by_strategy[\"{strategy}\"]는 vectors, bm25 두 키의 dict입니다."
    # 그 전략의 검색기가 실제로 실행하는 검색기(hybrid는 retrievers 안의 둘)
    parts = [retriever]
    if isinstance(retriever, EnsembleRetriever):
        parts = retriever.retrievers
    runs_dense = False
    runs_bm25 = False
    for part in parts:
        if part is dense:
            runs_dense = True
        if part is bm25:
            runs_bm25 = True
    assert uses["vectors"] is runs_dense, f"{strategy}: 이 전략의 검색기가 벡터 인덱스(dense)를 쓰는지 2-1에서 다시 보세요."
    assert uses["bm25"] is runs_bm25, f"{strategy}: 이 전략의 검색기가 BM25 색인(bm25)을 쓰는지 2-1에서 다시 보세요."

assert isinstance(accuracy_by_strategy, dict) and set(accuracy_by_strategy) == set(strategies), "accuracy_by_strategy의 키는 bm25, dense, hybrid입니다."
for strategy in strategies:
    assert accuracy_by_strategy[strategy] == overall[strategy], f"accuracy_by_strategy[\"{strategy}\"]는 overall의 그 전략 값입니다."

assert isinstance(candidates, pd.DataFrame), "candidates에 build_candidates의 결과를 담으세요."
assert list(candidates["candidate"]) == ["A", "B", "C"], "후보는 A, B, C 세 행입니다."
expected_candidates = build_candidates(uses_by_strategy, accuracy_by_strategy, ASSUMED_DOCS, machine_usd_per_hour)
assert candidates.equals(expected_candidates), "build_candidates에 품목 수 ASSUMED_DOCS와 장비 가격 machine_usd_per_hour를 넘기세요."
print("후보표 확인 완료")

**결과 읽기**
- 정확도는 A(`dense`) 0.555, B(`hybrid`) 0.546, C(`bm25`) 0.398 순입니다.
- A·B는 질문 임베딩 장비를 늘 켜 두어 월 비용이 약 $255.5이고, C는 색인 비용만 남아 $0.002입니다.
- 지연은 A 90ms, B 91ms, C 31ms이고, 저장 공간은 벡터를 쓰는 A(0.614GB)·B(0.697GB)가 BM25만 쓰는 C(0.083GB)보다 큽니다.
- B는 A보다 정확도가 낮고 지연·비용·저장 공간은 같거나 큽니다. 6번의 지배 확인에서 이 관계를 확인합니다.

## 6. 제약·지배·점수로 고르고 환율을 검산하세요

후보표가 생겼으니 교안의 세 단계(제약으로 거르기, 지배당한 후보 지우기, 고정 기준 점수)로 1위를 고릅니다. 팀이 합의한 가중치를 그대로 믿기 전에 환율로 그 뜻을 확인합니다.

| 제공 코드 | 입력과 반환값 |
|---|---|
| `apply_constraints(candidates, limits)` | 통과한 후보 표, 탈락한 후보와 이유(`drop_reason`) 표 |
| `remove_dominated(candidates, axes)` | 지배당한 후보를 지운 표, `{지배당한 후보: [이긴 후보 목록]}` |
| `score_candidates(candidates, weights, references)` | 축 점수와 가중 합 `score`를 담은 점수표(점수 높은 순) |
| `run_selection(candidates, limits, weights, references)` | 세 단계를 실행해 `dropped`, `dominated`, `scored`, `winner`, `gap`(1위와 2위의 점수 차)을 담은 dict |
| `exchange_rates(weights, references)` | 정확도 1%p와 같은 점수가 되는 각 축의 양(지연 ms, 월 비용 $, 저장 GB) dict |

In [ ]:
def apply_constraints(candidates, limits):
    """한도를 넘지 않은 후보 표와, 넘은 후보에 이유(drop_reason)를 붙인 표 두 개를 반환합니다."""
    passed_rows = []
    dropped_rows = []

    # 후보마다 한도(상한)를 하나씩 확인해 탈락 이유 모으기
    for row in candidates.to_dict("records"):
        reasons = []
        for column, limit in limits.items():
            if row[column] > limit:
                reasons.append(f"{column}>{limit}")

        # 이유가 하나라도 있으면 탈락
        if reasons:
            row["drop_reason"] = ", ".join(reasons)
            dropped_rows.append(row)
        else:
            passed_rows.append(row)

    # 탈락 후보가 없어도 candidate 열이 있도록 열 이름 지정
    return pd.DataFrame(passed_rows), pd.DataFrame(dropped_rows, columns=list(candidates.columns) + ["drop_reason"])


# 네 축과 좋은 방향
axes = {"accuracy": "max", "latency_ms": "min", "cost_usd": "min", "storage_gb": "min"}


def remove_dominated(candidates, axes):
    """지배당한 후보를 지운 표와 {지배당한 후보: [이긴 후보 목록]} dict를 반환합니다."""
    rows = candidates.to_dict("records")
    kept = []
    dominated = {}

    # 후보(row)마다 다른 모든 후보(other)와 비교해 row를 지배하는 후보 모으기
    for row in rows:
        winners = []
        for other in rows:
            if other["candidate"] == row["candidate"]:
                continue
            no_worse = True   # other가 모든 축에서 row보다 같거나 좋은가
            better = False    # other가 한 축 이상에서 row보다 좋은가
            for axis, direction in axes.items():
                # 차이가 양수면 other가 그 축에서 좋음(작을수록 좋은 축은 부호를 뒤집음)
                difference = other[axis] - row[axis]
                if direction == "min":
                    difference = -difference
                if difference < 0:
                    no_worse = False
                if difference > 0:
                    better = True
            if no_worse and better:
                winners.append(other["candidate"])

        # 이긴 후보가 있으면 지배당한 후보, 없으면 남김
        if winners:
            dominated[row["candidate"]] = winners
        else:
            kept.append(row)
    return pd.DataFrame(kept), dominated


def score_candidates(candidates, weights, references):
    """축마다 0~1 점수로 바꾸고 가중치를 곱해 더한 점수표(점수 높은 순)를 반환합니다."""
    rows = []
    # 후보마다 축 점수에 가중치를 곱해 더하기
    for row in candidates.to_dict("records"):
        scored = {"candidate": row["candidate"]}
        total = 0.0
        for axis, weight in weights.items():
            if axis in references:
                # 작을수록 좋은 축: 기준보다 작을수록 1에 가깝고, 기준을 넘으면 0점
                value = 1 - row[axis] / references[axis]
                if value < 0:
                    value = 0.0
            else:
                # 정확도는 이미 0~1
                value = row[axis]
            scored[axis + " 점수"] = round(value, 3)
            total += weight * value

        # 합계는 반올림하지 않아 격차를 정확히 계산함(화면에서만 반올림)
        scored["score"] = total
        rows.append(scored)

    # 점수 높은 순, 동점이면 후보 이름 순
    return pd.DataFrame(rows).sort_values(["score", "candidate"], ascending=[False, True]).reset_index(drop=True)


def run_selection(candidates, limits, weights, references):
    """제약 -> 지배 제거 -> 고정 기준 점수를 실행해 단계별 결과와 1위·격차를 dict로 반환합니다."""
    passed, dropped = apply_constraints(candidates, limits)
    front, dominated = remove_dominated(passed, axes)
    scored = score_candidates(front, weights, references)

    # 1위와 2위의 점수 차(0.001보다 작은 차이도 보이게 소수 넷째 자리까지, 후보가 하나면 None)
    gap = None
    if len(scored) > 1:
        gap = round(float(scored.loc[0, "score"] - scored.loc[1, "score"]), 4)
    return {"dropped": dropped, "dominated": dominated, "scored": scored, "winner": scored.loc[0, "candidate"], "gap": gap}


def exchange_rates(weights, references):
    """정확도 1%p와 같은 점수가 되는 각 축의 양(지연 ms, 월 비용 $, 저장 GB)을 dict로 반환합니다."""
    rates = {}
    # 정확도 1%p가 올리는 점수(0.01 x 정확도 가중치)를 축 값 1이 깎는 점수(축 가중치 / 기준)로 나눔
    for axis in references:
        rates[axis] = round(0.01 * weights["accuracy"] / (weights[axis] / references[axis]), 3)
    return rates

**배경**: 선정 기준의 고정값을 함수가 읽는 dict로 옮겨야 세 단계를 실행할 수 있고, 가중치는 환율로 바꿔 봐야 팀이 무엇을 무엇과 바꾸기로 했는지 읽을 수 있습니다.

**요구사항**
- **`limits`**: dict. 선정 기준의 제약 두 개(키 `latency_ms`, `storage_gb`, 값은 상한)
- **`references`**: dict. 고정 기준 세 개(키 `latency_ms`, `cost_usd`, `storage_gb`, 값은 실수)
- **`weights`**: dict. 가중치 네 개(키 `accuracy`, `latency_ms`, `cost_usd`, `storage_gb`, 합 1)
- **`rates`**: dict. `exchange_rates(weights, references)`의 결과
- **`result`**: dict. `run_selection(candidates, limits, weights, references)`의 결과

**확인 기준**: 자가채점 아래 셀이 환율, 제약 탈락, 지배당한 후보, 1위와 격차, 점수표를 출력합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 선정 기준의 '제약', '고정 기준', '가중치' 줄을 그대로 dict로 옮기고 제공 함수 두 개를 부른다.

세부구현:
1. 제약 줄의 두 값을 축 이름 키로 limits에 담는다.
2. 고정 기준 줄의 세 값을 실수로 references에 담는다.
3. 가중치 줄의 네 값을 weights에 담고 합이 1인지 본다.
4. exchange_rates와 run_selection에 위 dict와 후보표를 넘긴다.
```

</details>

In [ ]:
# [작성] 선정 기준의 고정값을 dict로 옮기고 선정을 실행하세요.
# 1. 제약 두 개
limits = ...

# 2. 고정 기준 세 개(받아들일 수 있는 최대치)
references = ...

# 3. 가중치 네 개(합 1)
weights = ...

# 4. 환율
rates = ...

# 5. 제약 -> 지배 제거 -> 고정 기준 점수
result = ...

In [ ]:
# [자가채점] 세 dict가 선정 기준과 같은지(지문), 환율, 선정 결과가 그 값으로 실행한 것인지 확인합니다.
# dict마다 "키=실수"를 키 순서로 이은 문자열의 지문(값을 그대로 적지 않으려고 sha256 앞 12글자만 둠)
settings_fingerprints = {"limits": "c3aa1edf101a", "references": "683ff1ec8431", "weights": "5fd94490daf2"}
for name, values in [("limits", limits), ("references", references), ("weights", weights)]:
    assert isinstance(values, dict), f"{name}에 dict를 담으세요. `...`를 채웠는지도 확인하세요."
    text = "|".join(f"{key}={float(values[key])}" for key in sorted(values))
    digest = hashlib.sha256(text.encode("utf-8")).hexdigest()[:12]
    assert digest == settings_fingerprints[name], f"{name}: 선정 기준의 키 이름과 값을 다시 보세요(키는 요구사항에 적힌 이름 그대로)."
assert abs(sum(weights.values()) - 1) < 1e-9, "weights의 합은 1입니다."
assert isinstance(rates, dict) and rates == exchange_rates(weights, references), "rates에는 exchange_rates(weights, references)의 결과를 담으세요."

# 선정 결과의 키와 1위
assert isinstance(result, dict) and set(result) == {"dropped", "dominated", "scored", "winner", "gap"}, "result에는 run_selection의 결과 dict를 담으세요."
assert result["winner"] == result["scored"].loc[0, "candidate"], "result는 run_selection이 돌려준 dict 그대로입니다."

# 제약: 한도를 넘는 후보만 탈락했는지(run_selection에 candidates와 limits를 넘겼는지)
dropped_names = list(result["dropped"]["candidate"])
for row in candidates.to_dict("records"):
    is_over = False
    for column, limit in limits.items():
        if row[column] > limit:
            is_over = True
    assert (row["candidate"] in dropped_names) == is_over, "run_selection에 후보표 candidates와 limits를 넘기세요."

# 점수: 축 점수가 references로 매긴 값이고, 합계가 weights로 매긴 값인지
candidate_rows = candidates.set_index("candidate")
for row in result["scored"].to_dict("records"):
    for axis, reference in references.items():
        axis_score = 1 - float(candidate_rows.loc[row["candidate"], axis]) / reference  # float로 바꿔 파이썬 round와 같은 규칙으로 반올림
        if axis_score < 0:
            axis_score = 0.0
        assert row[axis + " 점수"] == round(axis_score, 3), f"{row['candidate']}: {axis} 점수가 references로 매긴 값과 다릅니다. run_selection에 references를 넘기세요."
    weighted = 0.0
    for axis, weight in weights.items():
        weighted += weight * row[axis + " 점수"]
    # 축 점수는 소수 셋째 자리로 반올림돼 있어 작은 오차는 허용
    assert abs(row["score"] - weighted) < 0.003, "run_selection에 weights를 넘기세요."
print("선정 확인 완료")

In [ ]:
# 환율과 단계별 결과
print("정확도 1%p와 같은 점수:", rates)
print("제약 탈락:", list(result["dropped"]["candidate"]), "| 지배당함(이긴 후보):", result["dominated"])
print("1위:", result["winner"], "| 격차:", result["gap"])
display(result["scored"].round(3))

**결과 읽기**
- 환율은 정확도 1%p가 지연 35ms, 월 비용 $56, 저장 0.14GB와 같다는 뜻입니다.
- 세 후보 모두 제약을 통과하고, B는 A에게 지배당해 빠집니다.
- 1위는 A(0.608), 2위는 C(0.568)이고 격차는 0.0396입니다.
- C는 A보다 월 비용이 약 $255.5(환율로 약 4.6%p), 지연이 59ms(약 1.7%p), 저장 공간이 0.531GB(약 3.8%p) 작지만, 정확도가 15.7%p 낮아 세 이점을 더한 값(약 10.0%p)보다 큽니다.

## 7. 가정을 바꾼 시나리오로 다시 고르세요

1위는 기본 시나리오의 가정 위에서 나온 결과라, 어느 가정이 1위를 바꾸는지 알아야 ADR-004에 다시 검토할 조건을 적을 수 있습니다.

| 시나리오 | 바꾸는 가정 | `build_candidates`에 넘길 품목 수와 장비 가격 |
|---|---|---|
| `기본` | 선정 기준 그대로 | `ASSUMED_DOCS`, `machine_usd_per_hour` |
| `(a) 질문 임베딩 장비 비용 없음` | 질문 임베딩과 색인을 이미 있는 검색 서버에서 해 장비를 따로 빌리지 않음 | `ASSUMED_DOCS`, 0.0 |
| `(b) 품목 50만 건` | 취급 품목이 열 배로 늘어남 | 500,000, `machine_usd_per_hour` |
| `(c) 장비 가격 두 배` | 질문 임베딩 장비가 시간당 $0.70 | `ASSUMED_DOCS`, 0.70 |

**배경**: 품목 수와 장비 가격은 저장 공간과 비용 축을 바꿔 제약 통과와 점수에 영향을 주므로, 가정 하나씩만 바꾼 후보표로 선정을 다시 실행해 1위와 격차를 비교합니다.

**요구사항**
- **`scenario_tables`**: dict. 키는 위 표의 시나리오 이름(그 순서), 값은 그 시나리오의 품목 수·장비 가격으로 `build_candidates(uses_by_strategy, accuracy_by_strategy, 품목 수, 장비 가격)`를 실행한 후보표
- **`scenario_rows`**: list[dict]. 시나리오마다 그 후보표와 `limits`, `weights`, `references`로 `run_selection`을 실행해 한 행씩. 열은 아래 순서입니다.

| 열 | 값 |
|---|---|
| `시나리오` | 시나리오 이름 |
| `제약 탈락` | 탈락한 후보 이름 리스트 |
| `지배당함` | 지배당한 후보 이름 리스트 |
| `파레토 후보` | 점수표의 후보 이름 리스트(점수 높은 순) |
| `1위`, `격차` | 결과의 `winner`, `gap` |

**확인 기준**: `기본` 행은 6번의 `result`와 1위·격차가 같습니다. 후보가 하나만 남으면 격차는 `None`입니다(표로 출력하면 `NaN`으로 보입니다).

<details><summary>힌트</summary>

```text
접근방법:
- 시나리오 이름과 (품목 수, 장비 가격)을 짝지은 목록을 돌며 후보표를 만들고, 같은 반복에서 선정을 실행해 한 행씩 모은다.

세부구현:
1. (이름, 품목 수, 장비 가격) 튜플 네 개를 표 순서대로 리스트에 담는다.
2. 반복마다 build_candidates로 후보표를 만들어 scenario_tables[이름]에 넣는다.
3. 같은 후보표와 limits, weights, references로 run_selection을 실행한다.
4. 결과의 dropped 표 candidate 열, dominated의 키, scored 표 candidate 열을 list로 바꾸고 winner·gap과 함께 한 행으로 더한다.
```

</details>

In [ ]:
# [작성] 가정을 하나씩 바꾼 후보표로 선정을 다시 실행하세요.
# 1. 시나리오마다 (이름, 품목 수, 장비 가격)
scenario_settings = ...

# 2. 시나리오마다 후보표를 만들고 선정을 다시 실행해 한 행씩 모으기
scenario_tables = {}
scenario_rows = []
...

In [ ]:
# [자가채점] 시나리오별 후보표가 가정 하나만 바꿨는지, 결과 표의 모양과 시나리오마다 성립해야 하는 관계를 확인합니다.
expected_labels = ["기본", "(a) 질문 임베딩 장비 비용 없음", "(b) 품목 50만 건", "(c) 장비 가격 두 배"]
assert isinstance(scenario_tables, dict) and list(scenario_tables) == expected_labels, f"scenario_tables의 키는 이 순서입니다: {expected_labels}. `...`를 채웠는지도 확인하세요."
base_table = scenario_tables["기본"]
assert base_table.equals(candidates), "기본 시나리오의 후보표는 5번의 candidates와 같아야 합니다(ASSUMED_DOCS, machine_usd_per_hour)."

# (a) 장비 가격 0이면 비용이 모두 0, 다른 축은 그대로
table_a = scenario_tables["(a) 질문 임베딩 장비 비용 없음"]
assert (table_a["cost_usd"] == 0).all(), "(a)는 장비 가격 0.0으로 만드세요. 색인과 질문 임베딩 비용이 모두 0이 됩니다."
assert table_a["storage_gb"].equals(base_table["storage_gb"]), "(a)는 품목 수를 바꾸지 않습니다."

# (b) 품목이 열 배면 저장 공간이 약 열 배, 장비 월 비용은 그대로
table_b = scenario_tables["(b) 품목 50만 건"]
for position in base_table.index:
    assert abs(table_b.loc[position, "storage_gb"] - 10 * base_table.loc[position, "storage_gb"]) < 0.01, "(b)는 품목 수 500,000으로 만드세요."
assert (table_b["cost_usd"] >= base_table["cost_usd"]).all(), "(b)는 장비 가격을 바꾸지 않습니다."

# (c) 장비 가격이 두 배면 비용이 약 두 배, 저장 공간은 그대로
table_c = scenario_tables["(c) 장비 가격 두 배"]
for position in base_table.index:
    assert abs(table_c.loc[position, "cost_usd"] - 2 * base_table.loc[position, "cost_usd"]) < 0.01, "(c)는 장비 가격 0.70으로 만드세요."
assert table_c["storage_gb"].equals(base_table["storage_gb"]), "(c)는 품목 수를 바꾸지 않습니다."

# 결과 표: 네 행, 열, 시나리오마다 후보 셋이 탈락·지배·파레토 가운데 하나
assert isinstance(scenario_rows, list) and len(scenario_rows) == 4, "scenario_rows에 시나리오 넷의 결과를 한 행(dict)씩 담으세요."
for label, row in zip(expected_labels, scenario_rows):
    assert list(row) == ["시나리오", "제약 탈락", "지배당함", "파레토 후보", "1위", "격차"], "열은 시나리오, 제약 탈락, 지배당함, 파레토 후보, 1위, 격차 순서입니다."
    assert row["시나리오"] == label, f"시나리오 이름과 순서: {expected_labels}"
    assert row["1위"] == row["파레토 후보"][0], f"{label}: 1위는 점수표 맨 위 후보입니다."
    assert sorted(row["제약 탈락"] + row["지배당함"] + row["파레토 후보"]) == ["A", "B", "C"], f"{label}: 탈락·지배·파레토 후보를 합하면 세 후보입니다."
    recheck = run_selection(scenario_tables[label], limits, weights, references)
    assert row["1위"] == recheck["winner"] and row["격차"] == recheck["gap"], f"{label}: 그 시나리오의 후보표와 limits, weights, references로 run_selection을 실행하세요."
    assert sorted(row["제약 탈락"]) == sorted(recheck["dropped"]["candidate"]) and sorted(row["지배당함"]) == sorted(recheck["dominated"]), f"{label}: 제약 탈락은 결과 dropped의 후보 이름, 지배당함은 결과 dominated의 후보 이름입니다."
    assert list(row["파레토 후보"]) == list(recheck["scored"]["candidate"]), f"{label}: 파레토 후보는 점수표(scored)의 후보 이름을 점수 높은 순 그대로 담습니다."
assert scenario_rows[0]["1위"] == result["winner"] and scenario_rows[0]["격차"] == result["gap"], "기본 시나리오는 6번의 result와 같아야 합니다."
print("시나리오 확인 완료")

In [ ]:
# 시나리오별 결과
display(pd.DataFrame(scenario_rows))

**결과 읽기**
- (a) 장비 비용이 없으면 A·B의 비용 차이가 사라져 A의 격차가 0.0716으로 커집니다.
- (b) 품목이 50만 건이면 벡터를 쓰는 A·B의 저장 공간이 열 배로 늘어 저장 2GB 제약에서 탈락하고 C만 남습니다(격차 없음). 1위를 바꾸는 가정은 품목 수입니다.
- (c) 장비 가격이 두 배면 A·B의 월 비용이 두 배로 늘어 격차가 0.0077로 줄지만 1위는 그대로 A입니다. 1위를 바꾸지는 않지만 격차가 기본의 약 5분의 1이라 장비 가격이 더 오르면 C가 앞설 수 있습니다.

## 8. 1위를 뒤집는 가중치를 찾으세요

시나리오는 가정을 바꿔 봤으니, 이번에는 팀이 합의한 가중치가 달라지면 1위가 바뀌는지 봅니다. 교안의 ±20%보다 넓게, 가중치 하나를 `FLIP_FACTORS`의 배수로 바꿔(합은 다시 1로) 1위가 바뀌는 조합을 찾습니다.

| 제공 코드 | 입력과 반환값 |
|---|---|
| `FLIP_FACTORS` | 가중치에 곱할 배수 일곱 개 |
| `check_sensitivity(candidates, limits, weights, references, factors=(0.8, 1.2))` | 가중치 하나씩 `factors`의 배수로 바꾼 조합마다 `changed_weight`(예: `accuracy x0.5`), `winner`, `gap`을 담은 표 |

In [ ]:
# 가중치에 곱할 배수(교안의 0.8·1.2보다 넓게)
FLIP_FACTORS = (0.5, 0.75, 1.25, 1.5, 2.0, 3.0, 5.0)


def check_sensitivity(candidates, limits, weights, references, factors=(0.8, 1.2)):
    """가중치 하나씩 factors의 배수로 바꾼 조합마다 1위와 격차를 담은 표를 반환합니다."""
    rows = []
    # 가중치 하나를 factors의 배수로 바꾼 조합마다 선정 다시 실행
    for axis in weights:
        for factor in factors:
            changed = dict(weights)
            changed[axis] = weights[axis] * factor

            # 합으로 나눠 가중치 합을 다시 1로
            total = sum(changed.values())
            for name in changed:
                changed[name] = changed[name] / total

            result = run_selection(candidates, limits, changed, references)
            rows.append({"changed_weight": f"{axis} x{factor}", "winner": result["winner"], "gap": result["gap"]})
    return pd.DataFrame(rows)

**배경**: 1위가 가중치를 조금만 바꿔도 뒤집히면 그 결정은 팀 합의값에 기대고 있으므로, 넓은 배수로 바꿔 보며 어느 축을 몇 배로 볼 때 1위가 바뀌는지 찾습니다.

**요구사항**
- **`sensitivity`**: DataFrame(가중치 4개 × 배수 7개 = 28행). 기본 시나리오(`candidates`, `limits`, `weights`, `references`)로 `check_sensitivity`를 실행하되 `factors`에 `FLIP_FACTORS`를 넘긴 결과
- **`flip_rows`**: DataFrame. `sensitivity`에서 `winner`가 기본 1위(`result["winner"]`)와 다른 행만 거른 표(행 순서 유지)

**확인 기준**: 자가채점 셀이 1위가 바뀐 조합 수를 출력합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 제공 함수 하나를 기본 시나리오 값과 넓은 배수로 부르고, 결과 표를 winner 열 조건으로 거른다.

세부구현:
1. check_sensitivity에 기본 시나리오의 네 값을 넘기고 factors 인자에 FLIP_FACTORS를 준다.
2. sensitivity["winner"]가 result["winner"]와 다른 행을 불리언 조건으로 거른다.
```

</details>

In [ ]:
# [작성] 가중치를 넓은 배수로 바꿔 1위가 바뀌는 조합을 찾으세요.
# 1. 기본 시나리오에서 가중치 하나씩 FLIP_FACTORS 배수로 바꿔 보기
sensitivity = ...

# 2. 1위가 기본 1위와 다른 조합만 남기기
flip_rows = ...
display(flip_rows)

In [ ]:
# [자가채점] 민감도 표의 행·열과, 1위가 바뀐 행만 정확히 골랐는지 확인합니다.
assert isinstance(sensitivity, pd.DataFrame), "sensitivity에 check_sensitivity의 결과를 담으세요. `...`를 채웠는지도 확인하세요."
assert list(sensitivity.columns) == ["changed_weight", "winner", "gap"], "sensitivity는 check_sensitivity가 돌려준 표 그대로입니다."
assert len(sensitivity) == len(weights) * len(FLIP_FACTORS), "factors에 FLIP_FACTORS를 넘기세요. 가중치 4개 x 배수 7개 = 28행입니다."

# 첫 행 하나만 그 가중치로 선정을 다시 실행해 기본 시나리오 값으로 실행했는지 확인
first_axis, first_factor = sensitivity.iloc[0]["changed_weight"].split(" x")
first_weights = dict(weights)
first_weights[first_axis] = weights[first_axis] * float(first_factor)
first_total = sum(first_weights.values())
for name in first_weights:
    first_weights[name] = first_weights[name] / first_total
first_rerun = run_selection(candidates, limits, first_weights, references)
assert sensitivity.iloc[0]["winner"] == first_rerun["winner"] and sensitivity.iloc[0]["gap"] == first_rerun["gap"], "기본 시나리오의 candidates, limits, weights, references로 실행하세요."

# flip_rows: 남긴 행은 모두 1위가 바뀐 행, 뺀 행은 모두 기본 1위 그대로
assert isinstance(flip_rows, pd.DataFrame), "flip_rows에 거른 DataFrame을 담으세요."
assert list(flip_rows.index) == sorted(set(flip_rows.index) & set(sensitivity.index)), "flip_rows는 sensitivity에서 행을 거르기만 하고 순서는 그대로 두세요."
for position in sensitivity.index:
    is_flipped = sensitivity.loc[position, "winner"] != result["winner"]
    if position in flip_rows.index:
        assert is_flipped, "flip_rows에는 winner가 기본 1위(result[\"winner\"])와 다른 행만 남기세요."
    else:
        assert not is_flipped, "winner가 기본 1위와 다른 행은 빠짐없이 flip_rows에 남기세요."
print("민감도 확인 완료 | 1위가 바뀐 조합:", len(flip_rows), "개")

1위가 바뀐 첫 조합의 가중치(합을 1로 다시 맞춘 값)로 환율을 다시 구해 기본 환율과 비교합니다.

In [ ]:
# 1위가 바뀐 첫 조합의 축과 배수
axis, factor = flip_rows.iloc[0]["changed_weight"].split(" x")

# check_sensitivity와 같은 방법으로 그 축만 배수를 곱하고 합을 1로 맞춤
flip_weights = dict(weights)
flip_weights[axis] = weights[axis] * float(factor)
total = sum(flip_weights.values())
for name in flip_weights:
    flip_weights[name] = flip_weights[name] / total

print("바뀐 조합:", flip_rows.iloc[0]["changed_weight"], "| 1위:", flip_rows.iloc[0]["winner"], "| 격차:", flip_rows.iloc[0]["gap"])
print("합을 1로 맞춘 가중치:", {name: round(value, 3) for name, value in flip_weights.items()})
print("기본 환율:", rates)
print("바뀐 조합의 환율:", exchange_rates(flip_weights, references))

**결과 읽기**
- 28조합 가운데 1위가 바뀐 조합은 6개이고 모두 C로 바뀝니다. ±25%(배수 0.75·1.25)로는 바뀌지 않습니다.
- 처음 바뀌는 조합은 정확도 가중치 0.5배(합을 1로 맞추면 0.538)이고 격차는 0.0236입니다. 이때 환율은 정확도 1%p가 지연 17.5ms, 월 $28, 저장 0.07GB로 기본의 절반이 되어, A가 C보다 불리한 세 축(비용 약 9.1%p, 지연 약 3.4%p, 저장 약 7.6%p)을 더한 값이 약 20.1%p로 정확도 차이 15.7%p보다 커집니다.
- 비용·저장 가중치 3배·5배, 지연 가중치 5배에서도 C가 1위가 됩니다.

## 최종 아키텍처 ADR-004를 쓰세요

앞 ADR(과제 01 ADR-001, ADR-002·003) 위에서 고른 최종 구성과, 그 결정을 뒤집을 가정을 남깁니다. 아래 셀은 과제 01의 결정과 이 과제의 전략·저장 형식·저장소를 출력합니다.

In [ ]:
# 과제 01의 모델과 이 과제의 전략·저장 형식·저장소(ADR-004의 결정 칸에 옮김)
print("과제 01 임베딩 모델:", embedding_choice["model"], "| 저장 형식:", chosen_precision)
print("검색 질문 전략:", best_single, "| 라우터:", use_router, "| 구조화 질의 유형:", [name for name, path in answer_path.items() if path == "구조화 질의"])
print("저장소:", chosen_store, "| 네 축 1위:", result["winner"])

**작성:** 아래 마크다운 셀을 더블클릭해 괄호 안 안내를 지우고 제목과 네 칸(상태·맥락·결정·결과)을 채웁니다. 제목은 고른 기술이 드러나게 씁니다(예: "ADR-00N: ○○로 △△ 사용"). 맥락은 정해야 할 문제·제약·후보를 2~3문장으로 쓰고 결정은 쓰지 않습니다. 결정은 "…를 쓴다."로 시작하고, 아래 `근거:`에 검토한 선택지마다 한 줄씩 버린 이유를 쓴 뒤 고른 것이 이긴 이유를 씁니다. 결과 아래에는 `얻는 것:`, `감수하는 것:`, `다시 검토할 조건:`(어떤 가정이나 수치가 바뀌면 이 결정을 다시 내려야 하는지: 바뀌는 것 -> 다시 할 일)을 씁니다. ADR은 서비스 문서처럼 교안·과제·절 번호 없이 사실을 직접 씁니다. 숫자는 자신이 실행한 출력에서 옮깁니다.

**설명할 내용**
1. 상태: 제안·채택·거절·대체됨 가운데 하나(이 결정을 받아들였으면 `채택`)
2. 맥락: 정해야 할 것, 서비스와 규모, 선정 기준의 제약·고정 기준·가중치, 검토한 후보 A~C와 기존 ADR(ADR-001~003)
3. 결정: 최종 구성(임베딩 모델 + 검색 전략 + 저장소)과 기존 ADR 가운데 유지하는 것과 바꾸는 것을 "…를 쓴다."로 쓰고, `근거:`에 아래를 씁니다.
   - 후보 A~C마다 버린 이유(제약 탈락, 지배, 점수) 한 줄씩
   - 1위와 2위의 축 값 차이를 환율로 정확도 %p로 바꾼 비교
   - 시나리오 (a)·(b)·(c) 가운데 1위를 바꾸지 않은 가정
4. 결과: `얻는 것:`, `감수하는 것:`, `다시 검토할 조건:`을 3개 이상 쓰고, 아래를 모두 다룹니다.
   - 1위를 바꾼 시나리오의 가정
   - 1위를 뒤집는 가중치(어느 축을 몇 배로)와, 그때 환율이 기본 환율과 어떻게 달라졌는지
   - 정확도 축을 다시 측정해야 하는 경우(Self-Query 필터, 평가 질문 변경)

#### ADR-004: (고른 기술이 드러나는 제목)

- **상태**: (제안·채택·거절·대체됨 가운데 하나)
- **맥락**: (2~3문장: 정할 문제(최종 구성), 지켜야 할 조건(규모·제약·고정 기준·가중치, 기존 ADR-001~003), 검토한 후보(A~C). 결정은 쓰지 않음)
- **결정**: (고른 것을 한 문장으로: …를 쓴다)
    - 근거: (후보마다 한 줄: 뺀 이유(제약 탈락, 지배, 점수와 환율). 마지막 줄은 고른 것이 이긴 이유와 1위를 바꾸지 않은 시나리오)
- **결과**
    - 얻는 것: (이 결정으로 얻는 것)
    - 감수하는 것: (대신 감수하는 것)
    - 다시 검토할 조건: (바뀌는 가정이나 수치 -> 다시 할 일)